# South Africa JUSTICE project notebook


## Notebook structure

1. Setup and file locations
2. South Africa problem framing
3. Optimisation setup
4. Generate candidate policies
5. Search quality and reference set
6. Trade-offs, selected policies, and ECR interpretation
7. Robustness analysis and re-evaluation
8. Rival framing comparison
9. Report writing skeleton
10. Appendix: optional checks


## 0. Setup and file locations

In [1]:
# ============================================================
# 0. FLEXIBLE PROJECT SETUP

import os
import sys
from pathlib import Path
import importlib.util

def find_project_root(start_paths=None, required_folders=("justice", "config", "solvers")):
    """
    Find the project root by searching upward from possible start locations.
    The project root is the folder that directly contains:
    justice/, config/, and solvers/.
    """
    candidates = []

    # 1. Current working directory
    candidates.append(Path.cwd())

    # 2. Notebook directory, if available
    try:
        candidates.append(Path(__file__).resolve().parent)
    except NameError:
        pass

    # 3. Optional manually supplied paths
    if start_paths:
        candidates.extend(Path(p).expanduser().resolve() for p in start_paths)

    checked = []

    for start in candidates:
        start = start.expanduser().resolve()

        # Search current folder and all parents
        for folder in [start] + list(start.parents):
            checked.append(folder)

            if all((folder / req).exists() for req in required_folders):
                return folder

            # Also check common nested folders
            for subfolder_name in ["JUSTICE-main", "epa141a", "A- Project G15"]:
                nested = folder / subfolder_name
                if all((nested / req).exists() for req in required_folders):
                    return nested.resolve()

    raise FileNotFoundError(
        "Could not find the project root. "
        "Make sure your notebook is inside or near the folder containing "
        "'justice/', 'config/', and 'solvers/'."
    )


PROJECT_ROOT = find_project_root()

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Optional: set working directory to project root
os.chdir(PROJECT_ROOT)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())
print("Contains justice:", (PROJECT_ROOT / "justice").exists())
print("Contains config:", (PROJECT_ROOT / "config").exists())
print("Contains solvers:", (PROJECT_ROOT / "solvers").exists())

print("config:", importlib.util.find_spec("config"))
print("config.default_parameters:", importlib.util.find_spec("config.default_parameters"))
print("justice:", importlib.util.find_spec("justice"))

PROJECT_ROOT: /Users/juliusrupert/Documents/Uni/Y1Q4 EPA/Model-based Decision-making/epa141a/JUSTICE-main
Current working directory: /Users/juliusrupert/Documents/Uni/Y1Q4 EPA/Model-based Decision-making/epa141a/JUSTICE-main
Contains justice: True
Contains config: True
Contains solvers: True
config: ModuleSpec(name='config', loader=None, submodule_search_locations=_NamespacePath(['/Users/juliusrupert/Documents/Uni/Y1Q4 EPA/Model-based Decision-making/epa141a/JUSTICE-main/config', '/Users/juliusrupert/Documents/Uni/Y1Q4 EPA/Model-based Decision-making/epa141a/JUSTICE-main/config']))
config.default_parameters: ModuleSpec(name='config.default_parameters', loader=<_frozen_importlib_external.SourceFileLoader object at 0x187e27800>, origin='/Users/juliusrupert/Documents/Uni/Y1Q4 EPA/Model-based Decision-making/epa141a/JUSTICE-main/config/default_parameters.py')
justice: ModuleSpec(name='justice', loader=<_frozen_importlib_external.SourceFileLoader object at 0x187e278f0>, origin='/Users/juliu

In [2]:
# ============================================================
# 0. Imports and project folders
# Run this after the flexible PROJECT_ROOT setup cell
# ============================================================

import os
import sys
import json
from pathlib import Path
import warnings
import re
import shlex

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# PROJECT_ROOT komt uit de flexible setup cell
# Dit is de map waarin justice/, config/ en solvers/ staan.
_JUSTICE_ROOT = PROJECT_ROOT

# Zoek groeps/projectmap
# Optie 1: als A- Project G15 naast/in de project root staat
possible_project_dirs = [
    PROJECT_ROOT / "A- Project G15",
    PROJECT_ROOT.parent / "A- Project G15",
    PROJECT_ROOT,
]

_PROJECT_DIR = None
for candidate in possible_project_dirs:
    if candidate.exists():
        _PROJECT_DIR = candidate.resolve()
        break

if _PROJECT_DIR is None:
    _PROJECT_DIR = PROJECT_ROOT.resolve()

RESULTS_DIR = (_PROJECT_DIR / "results").resolve()
PLOTS_DIR = (_PROJECT_DIR / "plots").resolve()

for path in [RESULTS_DIR, PLOTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Sommige JUSTICE-code verwacht dat de working directory de project root is
os.chdir(_JUSTICE_ROOT)

from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction, Scenario

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Project dir:", _PROJECT_DIR)
print("JUSTICE root:", _JUSTICE_ROOT)
print("Results dir:", RESULTS_DIR)
print("Plots dir:", PLOTS_DIR)

PROJECT_ROOT: /Users/juliusrupert/Documents/Uni/Y1Q4 EPA/Model-based Decision-making/epa141a/JUSTICE-main
Project dir: /Users/juliusrupert/Documents/Uni/Y1Q4 EPA/Model-based Decision-making/epa141a/A- Project G15
JUSTICE root: /Users/juliusrupert/Documents/Uni/Y1Q4 EPA/Model-based Decision-making/epa141a/JUSTICE-main
Results dir: /Users/juliusrupert/Documents/Uni/Y1Q4 EPA/Model-based Decision-making/epa141a/A- Project G15/results
Plots dir: /Users/juliusrupert/Documents/Uni/Y1Q4 EPA/Model-based Decision-making/epa141a/A- Project G15/plots


In [21]:
# Basic model dimensions used in Assignment 4 and later optimisation.
# These are the standard values used in the assignment setup.
N_REGIONS = 57
N_INPUTS = 2
N_RBFS = N_INPUTS + 2
START_YEAR = 2015
END_YEAR = 2300
TIMESTEP = 1
DATA_TIMESTEP = 5
SMALL_NUMBER = 1e-9
# We get the exact region list from JUSTICE. This is light compared with full optimisation.
JUSTICE.hard_reset()
_model_probe = JUSTICE(
    start_year=START_YEAR,
    end_year=END_YEAR,
    timestep=TIMESTEP,
    scenario=2,
    climate_ensembles=1,
    stochastic_run=False,
    social_welfare_function=WelfareFunction.PRIORITARIAN,
)

REGION_LIST = list(_model_probe.data_loader.REGION_LIST)
N_REGIONS = len(REGION_LIST)
N_TIMESTEPS = len(_model_probe.time_horizon.model_time_horizon)
zaf_idx = REGION_LIST.index("zaf")

print("Number of regions:", N_REGIONS)
print("Number of timesteps:", N_TIMESTEPS)
print("South Africa index:", zaf_idx)
print("First 10 regions:", REGION_LIST[:10])

NameError: name 'JUSTICE' is not defined

## 1. South Africa problem framing

This section translates the actor mandate into model terms. It defines the region of interest, the preferred welfare lens, the XLRM framing, and the satisficing criteria.


### 1.1 Mandate translated to model terms

In [9]:
ACTOR = "South Africa"
KEY_REGION = "zaf"
PREFERRED_WELFARE = WelfareFunction.PRIORITARIAN
RIVAL_WELFARE = WelfareFunction.UTILITARIAN

south_africa_actor_frame = {
    "actor": ACTOR,
    "key_region": KEY_REGION,
    "preferred_welfare_function": PREFERRED_WELFARE.name,
    "rival_welfare_function": RIVAL_WELFARE.name,
    "core_metric": "abatement_cost / gross_economic_output",
    "main_claim": (
        "South Africa supports mitigation, but equal obligations are not equally fair "
        "when transition costs are unequal relative to economic capacity."
    ),
}

south_africa_actor_frame

{'actor': 'South Africa',
 'key_region': 'zaf',
 'preferred_welfare_function': 'PRIORITARIAN',
 'rival_welfare_function': 'UTILITARIAN',
 'core_metric': 'abatement_cost / gross_economic_output',
 'main_claim': 'South Africa supports mitigation, but equal obligations are not equally fair when transition costs are unequal relative to economic capacity.'}

### 1.2 XLRM framework for South Africa

This is the filled XLRM that connects the political mandate to model choices.

- **X — Uncertainties:** things South Africa cannot control.
- **L — Levers:** what the policy/search can change.
- **R — Relationships:** how JUSTICE turns policy into outcomes.
- **M — Metrics:** how we judge policy performance.

In [10]:
# ============================================================
# XLRM FRAMING — SOUTH AFRICA
# ============================================================

south_africa_xlrm = pd.DataFrame(
    [
        # X — EXOGENOUS UNCERTAINTIES
        {
            "XLRM": "X — Exogenous uncertainties",
            "Element": "Future socioeconomic pathway",
            "Meaning for South Africa": (
                "Future development affects emissions, economic output, climate damages, "
                "and South Africa's ability to carry transition costs."
            ),
            "Model operationalisation": (
                "Reference scenario SSP2-RCP4.5; broader scenario uncertainty is discussed "
                "as part of the political interpretation."
            ),
        },
        {
            "XLRM": "X — Exogenous uncertainties",
            "Element": "Climate response uncertainty",
            "Meaning for South Africa": (
                "The same mitigation policy can lead to different temperature outcomes "
                "depending on the climate response."
            ),
            "Model operationalisation": (
                "Subset of FaIR climate ensemble members used in the final runs."
            ),
        },
        {
            "XLRM": "X — Exogenous uncertainties",
            "Element": "Regional climate damages",
            "Meaning for South Africa": (
                "Insufficient mitigation can impose economic damages on South Africa, "
                "which must be weighed against the cost of abatement."
            ),
            "Model operationalisation": (
                "Damage outcomes for zaf, including damage fraction and economic damage "
                "relative to gross economic output."
            ),
        },
        {
            "XLRM": "X — Exogenous uncertainties",
            "Element": "Normative welfare framing",
            "Meaning for South Africa": (
                "The evaluation of a policy depends on whether worse-off regions receive "
                "additional weight or whether only aggregate global welfare is prioritised."
            ),
            "Model operationalisation": (
                "Prioritarian welfare as the main framing; Utilitarian welfare as the "
                "rival framing."
            ),
        },
        {
            "XLRM": "X — Exogenous uncertainties",
            "Element": "Availability and form of Just Transition finance",
            "Meaning for South Africa": (
                "Finance determines whether ambitious mitigation is politically and "
                "economically feasible for a coal-dependent emerging economy."
            ),
            "Model operationalisation": (
                "Not modelled directly; used to interpret whether South Africa's "
                "abatement burden is politically acceptable."
            ),
        },

        # L — POLICY LEVERS
        {
            "XLRM": "L — Policy levers",
            "Element": "Regional emission control rate",
            "Meaning for South Africa": (
                "Determines how quickly each region reduces emissions and therefore how "
                "much mitigation burden South Africa faces."
            ),
            "Model operationalisation": (
                "Emission control rate μ for each region over time."
            ),
        },
        {
            "XLRM": "L — Policy levers",
            "Element": "Adaptive mitigation policy",
            "Meaning for South Africa": (
                "Allows mitigation levels to respond over time instead of imposing one "
                "fixed pathway in advance."
            ),
            "Model operationalisation": (
                "RBF policy defined by centers, radii, and weights."
            ),
        },

        # R — RELATIONSHIPS
        {
            "XLRM": "R — Model relationships",
            "Element": "Climate-economy system",
            "Meaning for South Africa": (
                "Mitigation reduces climate risk but also creates abatement costs; both "
                "affect South Africa's economic outcomes."
            ),
            "Model operationalisation": (
                "JUSTICE links gross economic output, emissions, temperature, damages, "
                "abatement costs, consumption, and welfare."
            ),
        },
        {
            "XLRM": "R — Model relationships",
            "Element": "Burden mechanism",
            "Meaning for South Africa": (
                "The same mitigation effort can create unequal burdens because countries "
                "differ in economic capacity and energy-system dependence."
            ),
            "Model operationalisation": (
                "South Africa's burden is evaluated as abatement_cost divided by "
                "gross_economic_output for zaf."
            ),
        },

        # M — PERFORMANCE MEASURES
        {
            "XLRM": "M — Performance measures",
            "Element": "Global climate risk",
            "Meaning for South Africa": (
                "Measures whether the policy contributes to avoiding dangerous global warming."
            ),
            "Model operationalisation": (
                "fraction_above_threshold and global temperature in 2100."
            ),
        },
        {
            "XLRM": "M — Performance measures",
            "Element": "Global welfare",
            "Meaning for South Africa": (
                "Measures the overall welfare performance of a mitigation policy under "
                "the chosen welfare framing."
            ),
            "Model operationalisation": (
                "welfare, evaluated mainly under the Prioritarian welfare function."
            ),
        },
        {
            "XLRM": "M — Performance measures",
            "Element": "South Africa abatement burden",
            "Meaning for South Africa": (
                "Measures whether mitigation costs are disproportionate relative to "
                "South Africa's economic capacity."
            ),
            "Model operationalisation": (
                "abatement_cost / gross_economic_output for zaf."
            ),
        },
        {
            "XLRM": "M — Performance measures",
            "Element": "South Africa damage burden",
            "Meaning for South Africa": (
                "Measures the economic cost of insufficient mitigation for South Africa."
            ),
            "Model operationalisation": (
                "economic_damage / gross_economic_output and damage_fraction for zaf."
            ),
        },
        {
            "XLRM": "M — Performance measures",
            "Element": "South Africa net output protection",
            "Meaning for South Africa": (
                "Measures whether climate policy leaves South Africa with sufficient "
                "economic output after damages and abatement costs."
            ),
            "Model operationalisation": (
                "net_economic_output / gross_economic_output for zaf."
            ),
        },
    ]
)

display(south_africa_xlrm)

,XLRM,Element,Meaning for South Africa,Model operationalisation
0,X — Exogenous uncertainties,Future socioeconomic pathway,"Future development affects emissions, economic...",Reference scenario SSP2-RCP4.5; broader scenar...
1,X — Exogenous uncertainties,Climate response uncertainty,The same mitigation policy can lead to differe...,Subset of FaIR climate ensemble members used i...
2,X — Exogenous uncertainties,Regional climate damages,Insufficient mitigation can impose economic da...,"Damage outcomes for zaf, including damage frac..."
3,X — Exogenous uncertainties,Normative welfare framing,The evaluation of a policy depends on whether ...,Prioritarian welfare as the main framing; Util...
4,X — Exogenous uncertainties,Availability and form of Just Transition finance,Finance determines whether ambitious mitigatio...,Not modelled directly; used to interpret wheth...
5,L — Policy levers,Regional emission control rate,Determines how quickly each region reduces emi...,Emission control rate μ for each region over t...
6,L — Policy levers,Adaptive mitigation policy,Allows mitigation levels to respond over time ...,"RBF policy defined by centers, radii, and weig..."
7,R — Model relationships,Climate-economy system,Mitigation reduces climate risk but also creat...,"JUSTICE links gross economic output, emissions..."
8,R — Model relationships,Burden mechanism,The same mitigation effort can create unequal ...,South Africa's burden is evaluated as abatemen...
9,M — Performance measures,Global climate risk,Measures whether the policy contributes to avo...,fraction_above_threshold and global temperatur...


In [11]:
# ============================================================
# SAVE XLRM TABLE FOR OVERLEAF
# ============================================================

xlrm_latex = south_africa_xlrm.to_latex(
    index=False,
    escape=True,
    longtable=True,
    caption="XLRM framing for South Africa's just transition mandate",
    label="tab:xlrm_south_africa",
    column_format="p{3.2cm} p{4.0cm} p{6.0cm} p{6.0cm}",
)

output_path = PLOTS_DIR / "xlrm_south_africa_table.tex"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(xlrm_latex)

print(f"Saved LaTeX table to: {output_path}")

Saved LaTeX table to: /Users/wesselmuntendam/Model Based Decision Making/epa141a/A- Project G15/plots/xlrm_south_africa_table.tex


## 2. Global Sensitivy analysis

In [12]:
# ============================================================
# GLOBAL SENSITIVITY ANALYSIS — LAB 2 STYLE SETUP
# ============================================================

import os
import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from SALib.analyze import sobol as sobol_analyze

from ema_workbench import (
    Model,
    RealParameter,
    ScalarOutcome,
    Sample,
    MultiprocessingEvaluator,
    SequentialEvaluator,
    ema_logging,
    Samplers,
)

from ema_workbench.em_framework.salib_samplers import get_SALib_problem
from ema_workbench.analysis import feature_scoring

from justice.model import JUSTICE
from justice.util.enumerations import (
    Economy,
    DamageFunction,
    Abatement,
    WelfareFunction,
)
from justice.util.model_time import TimeHorizon
from justice.objectives.objective_functions import (
    years_above_temperature_threshold,
    fraction_of_ensemble_above_threshold,
)

ema_logging.log_to_stderr(logging.WARNING)

SMALL_NUMBER = 1e-9

GSA_DIR = os.path.join(RESULTS_DIR, "sobol_gsa")
os.makedirs(GSA_DIR, exist_ok=True)

GSA_PLOTS_DIR = os.path.join(PLOTS_DIR, "sobol_gsa")
os.makedirs(GSA_PLOTS_DIR, exist_ok=True)

GSA_OBJECTIVES = [
    "welfare",
    "years_above_temperature_threshold",
    "fraction_above_threshold",
    "global_temperature_2100",
    "max_global_temperature",
    "welfare_loss_damage",
    "welfare_loss_abatement",
    "zaf_mean_abatement_burden",
    "zaf_mean_damage_fraction",
    "zaf_mean_net_output_ratio",
]

GSA_PARAMS = ["rho", "eta", "delta", "ecs_ensemble"]

GSA_POLICY_NAMES = [
    "no_abatement",
    "moderate_abatement",
]

print("GSA setup ready")

GSA setup ready


In [13]:
# ============================================================
# LOAD JUSTICE CONFIG FOR GSA
# ============================================================

import os
import json
from pathlib import Path

# Common possible config locations
_possible_config_paths = [
    _PROJECT_DIR / "config" / "config_student.json",
    _PROJECT_DIR.parent / "config" / "config_student.json",
    Path.cwd() / "config" / "config_student.json",
    Path.cwd().parent / "config" / "config_student.json",
]

config_path = None

for path in _possible_config_paths:
    if path.exists():
        config_path = path
        break

if config_path is None:
    raise FileNotFoundError(
        "Could not find config_student.json. "
        "Check where your config folder is and set config_path manually."
    )

with open(config_path) as fh:
    _cfg = json.load(fh)

print("Loaded config from:", config_path)
print(_cfg)

Loaded config from: /Users/wesselmuntendam/Model Based Decision Making/epa141a/config/config_student.json
{'start_year': 2015, 'end_year': 2300, 'data_timestep': 5, 'timestep': 1, 'emission_control_start_year': 2025, 'n_rbfs': 4, 'n_inputs': 2, 'epsilons': [1.0, 0.01, 10.0, 10.0, 0.001], 'temperature_year_of_interest': 2100, 'reference_ssp_rcp_scenario_index': 2, 'stochastic_run': False, 'climate_ensemble_members': [85, 88, 93, 200, 429, 434, 524, 647, 693, 718, 735, 761, 764, 851, 973], 'social_welfare_function_type': 1}


In [ ]:
# ============================================================
# GLOBAL SENSITIVITY ANALYSIS — JUSTICE MODEL WRAPPER
# ============================================================
_gsa_time_horizon = TimeHorizon(
    start_year=_cfg["start_year"],
    end_year=_cfg["end_year"],
    data_timestep=_cfg["data_timestep"],
    timestep=_cfg["timestep"],
)

GSA_TEMP_YEAR_IDX = _gsa_time_horizon.year_to_timestep(
    year=_cfg["temperature_year_of_interest"],
    timestep=_cfg["timestep"],
)

GSA_SCENARIO = _cfg["reference_ssp_rcp_scenario_index"]


def _as_region_time_ensemble(arr):
    arr = np.asarray(arr)

    if arr.ndim == 2:
        return arr[:, :, None]

    if arr.ndim == 3:
        return arr

    raise ValueError(f"Unexpected region-time-ensemble shape: {arr.shape}")

def _as_time_ensemble(arr):
    arr = np.asarray(arr)

    if arr.ndim == 1:
        return arr[:, None]

    if arr.ndim == 2:
        return arr

    if arr.ndim == 3:
        return arr.mean(axis=0)

    raise ValueError(f"Unexpected time-ensemble shape: {arr.shape}")


def justice_gsa_model(
    rho=0.015,
    eta=1.45,
    delta=1.0,
    ecs_ensemble=501,
    ecr_plateau=0.0,
):
    """
    JUSTICE wrapper for Global Sensitivity Analysis.

    Uncertainties:
    - rho
    - eta
    - delta
    - ecs_ensemble

    Policy:
    - fixed emission control rate plateau
    """
    try:
        JUSTICE.hard_reset()

        ensemble_idx = int(np.round(np.clip(ecs_ensemble, 1, 1000)))

        model = JUSTICE(
            scenario=GSA_SCENARIO,
            climate_ensembles=[ensemble_idx],
            economy_type=Economy.NEOCLASSICAL,
            damage_function_type=DamageFunction.KALKUHL,
            abatement_type=Abatement.ENERDATA,
            social_welfare_function_type=WelfareFunction.PRIORITARIAN.value[0],
        )

        # Normative/economic uncertainties
        if hasattr(model.economy, "pure_rate_of_social_time_preference"):
            model.economy.pure_rate_of_social_time_preference = float(rho)

        if hasattr(model.economy, "elasticity_of_marginal_utility_of_consumption"):
            model.economy.elasticity_of_marginal_utility_of_consumption = float(eta)

        if hasattr(model.welfare_function, "pure_rate_of_social_time_preference"):
            model.welfare_function.pure_rate_of_social_time_preference = float(rho)

        if hasattr(model.welfare_function, "elasticity_of_marginal_utility_of_consumption"):
            model.welfare_function.elasticity_of_marginal_utility_of_consumption = float(eta)

        # Damage multiplier
        for attr in [
            "coefficient_a",
            "coefficient_b",
            "damage_gdp_ratio_with_gradient",
        ]:
            if hasattr(model.damage_function, attr):
                setattr(
                    model.damage_function,
                    attr,
                    getattr(model.damage_function, attr) * float(delta),
                )

        # Fixed policy: uniform ECR plateau across all regions and timesteps
        ecr = np.full(
            model.emission_control_rate.shape[:2],
            float(ecr_plateau),
        )

        model.run(
            emission_control_rate=ecr,
            endogenous_savings_rate=True,
        )

        data = model.evaluate()

        # Temperature
        global_temperature = _as_time_ensemble(data["global_temperature"])

        global_temperature_2100 = float(
            np.nanmean(global_temperature[GSA_TEMP_YEAR_IDX, :])
        )

        max_global_temperature = float(
            np.nanmax(global_temperature)
        )

        years_above = float(
            np.squeeze(
                years_above_temperature_threshold(
                    global_temperature,
                    threshold=2.0,
                )
            )
        )

        fraction_above = fraction_of_ensemble_above_threshold(
            temperature=global_temperature,
            temperature_year_index=GSA_TEMP_YEAR_IDX,
            threshold=2.0,
        )
        fraction_above = float(fraction_above)

        # Welfare
        welfare = float(np.abs(np.squeeze(data["welfare"])))

        damage_pc = np.maximum(data["damage_cost_per_capita"], SMALL_NUMBER)
        abatement_pc = np.maximum(data["abatement_cost_per_capita"], SMALL_NUMBER)

        _, _, _, wl_damage = model.welfare_function.calculate_welfare(
            damage_pc,
            welfare_loss=True,
        )

        _, _, _, wl_abatement = model.welfare_function.calculate_welfare(
            abatement_pc,
            welfare_loss=True,
        )

        welfare_loss_damage = float(np.abs(np.squeeze(wl_damage)))
        welfare_loss_abatement = float(np.abs(np.squeeze(wl_abatement)))

        # South Africa-specific outcomes
        gross_output = _as_region_time_ensemble(data["gross_economic_output"])
        net_output = _as_region_time_ensemble(data["net_economic_output"])
        abatement_cost = _as_region_time_ensemble(data["abatement_cost"])
        damage_fraction = _as_region_time_ensemble(data["damage_fraction"])

        region_list = list(model.region_list)
        zaf_idx = region_list.index("zaf")

        zaf_gross_output = gross_output[zaf_idx, :, :]
        zaf_net_output = net_output[zaf_idx, :, :]
        zaf_abatement_cost = abatement_cost[zaf_idx, :, :]
        zaf_damage_fraction_arr = damage_fraction[zaf_idx, :, :]

        zaf_mean_abatement_burden = float(
            np.nanmean(
                np.divide(
                    zaf_abatement_cost,
                    np.maximum(zaf_gross_output, SMALL_NUMBER),
                )
            )
        )

        zaf_mean_damage_fraction = float(
            np.nanmean(zaf_damage_fraction_arr)
        )

        zaf_mean_net_output_ratio = float(
            np.nanmean(
                np.divide(
                    zaf_net_output,
                    np.maximum(zaf_gross_output, SMALL_NUMBER),
                )
            )
        )

        return {
            "welfare": welfare,
            "years_above_temperature_threshold": years_above,
            "fraction_above_threshold": fraction_above,
            "global_temperature_2100": global_temperature_2100,
            "max_global_temperature": max_global_temperature,
            "welfare_loss_damage": welfare_loss_damage,
            "welfare_loss_abatement": welfare_loss_abatement,
            "zaf_mean_abatement_burden": zaf_mean_abatement_burden,
            "zaf_mean_damage_fraction": zaf_mean_damage_fraction,
            "zaf_mean_net_output_ratio": zaf_mean_net_output_ratio,
        }

    except Exception as e:
        print(f"[GSA FAILED RUN] {type(e).__name__}: {e}")

        return {
            "welfare": 1e6,
            "years_above_temperature_threshold": 1e6,
            "fraction_above_threshold": 1e6,
            "global_temperature_2100": 1e6,
            "max_global_temperature": 1e6,
            "welfare_loss_damage": 1e6,
            "welfare_loss_abatement": 1e6,
            "zaf_mean_abatement_burden": 1e6,
            "zaf_mean_damage_fraction": 1e6,
            "zaf_mean_net_output_ratio": -1e6,
        }


# Smoke test
test = justice_gsa_model()
for key, value in test.items():
    print(f"{key}: {value}")

In [ ]:
import time

t0 = time.time()
test = justice_gsa_model(
    rho=0.015,
    eta=1.0,
    delta=1.0,
    ecs_ensemble=501,
    ecr_plateau=0.4,
)
print(test)
print("Elapsed seconds:", time.time() - t0)

In [ ]:
# ============================================================
# GLOBAL SENSITIVITY ANALYSIS — EMA MODEL SETUP
# ============================================================

gsa_model = Model("JUSTICE_GSA", function=justice_gsa_model)

gsa_model.uncertainties = [
    RealParameter("rho", 0.001, 0.030),
    RealParameter("eta", 0.5, 1.5),
    RealParameter("delta", 0.5, 2.0),
    RealParameter("ecs_ensemble", 1, 1000),
]

# ecr_plateau is fixed through policy Samples, just like fixed release policies in Lab 2
gsa_model.levers = []

gsa_model.outcomes = [
    ScalarOutcome(outcome)
    for outcome in GSA_OBJECTIVES
]

gsa_policies = [
    Sample("no_abatement", ecr_plateau=0.0),
    Sample("moderate_abatement", ecr_plateau=0.4),
]

problem = get_SALib_problem(gsa_model.uncertainties)
print("SALib problem:")
print(problem)

In [ ]:
# ============================================================
# GLOBAL SENSITIVITY ANALYSIS — EMA MODEL SETUP FIX
# ============================================================

from ema_workbench import RealParameter, ScalarOutcome, Sample

gsa_model = Model("JUSTICE_GSA", function=justice_gsa_model)

gsa_model.uncertainties = [
    RealParameter("rho", 0.001, 0.030),
    RealParameter("eta", 0.5, 1.5),
    RealParameter("delta", 0.5, 2.0),
    RealParameter("ecs_ensemble", 1, 1000),
]

# IMPORTANT: define ecr_plateau as a lever, then fix it through policies
gsa_model.levers = [
    RealParameter("ecr_plateau", 0.0, 1.0),
]

gsa_model.outcomes = [
    ScalarOutcome(outcome)
    for outcome in GSA_OBJECTIVES
]

gsa_policies_smoke = [
    Sample("moderate_abatement", ecr_plateau=0.4),
]

problem = get_SALib_problem(gsa_model.uncertainties)

print(problem)

In [ ]:
# ============================================================
# SOBOL RUN — MODERATE ABATEMENT ONLY, SEQUENTIAL
# ============================================================

N_SOBOL = 16

gsa_policies = [
    Sample("moderate_abatement", ecr_plateau=0.4),
]

print(f"Running Sobol GSA with N={N_SOBOL}")

with SequentialEvaluator(gsa_model) as evaluator:
    experiments_sobol, outcomes_sobol = evaluator.perform_experiments(
        scenarios=N_SOBOL,
        policies=gsa_policies,
        uncertainty_sampling=Samplers.SOBOL,
    )

outcomes_sobol_df = pd.DataFrame(outcomes_sobol)

print("Experiments shape:", experiments_sobol.shape)
display(experiments_sobol.head())
display(outcomes_sobol_df.describe().round(4))

In [ ]:
# ============================================================
# SOBOL ANALYSIS — S1 AND ST
# ============================================================

sobol_results = {}

for policy_name in experiments_sobol["policy"].unique():
    mask = experiments_sobol["policy"] == policy_name

    sobol_results[policy_name] = {}

    for outcome in GSA_OBJECTIVES:
        y = outcomes_sobol_df.loc[mask, outcome].values.astype(float)

        finite = np.isfinite(y)

        if finite.sum() < len(y):
            y = np.where(finite, y, np.nanmedian(y[finite]))

        if np.nanstd(y) < 1e-12:
            print(f"Skipping {policy_name} — {outcome}: no output variance")
            sobol_results[policy_name][outcome] = None
            continue

        indices = sobol_analyze.analyze(
            problem,
            y,
            calc_second_order=True,
            print_to_console=False,
        )

        sobol_results[policy_name][outcome] = indices

print("Sobol analysis finished")

In [ ]:
# ============================================================
# SOBOL PLOTS — S1 AND ST PER OUTCOME
# ============================================================
import copy

SELECTED_GSA_OUTCOMES = [
    "global_temperature_2100",
    "welfare_loss_damage",
    "zaf_mean_abatement_burden",
    "zaf_mean_damage_fraction",
    "zaf_mean_net_output_ratio",
]

for outcome in SELECTED_GSA_OUTCOMES:
    fig, axes = plt.subplots(
        1,
        len(GSA_POLICY_NAMES),
        figsize=(12, 4),
        sharey=True,
    )

    if len(GSA_POLICY_NAMES) == 1:
        axes = [axes]

    for ax, policy_name in zip(axes, GSA_POLICY_NAMES):
        indices = sobol_results[policy_name][outcome]

        if indices is None:
            ax.set_title(f"{policy_name}\nno variance")
            ax.axis("off")
            continue

        si_df = pd.DataFrame(
            {
                "S1": indices["S1"],
                "ST": indices["ST"],
            },
            index=problem["names"],
        )

        err_df = pd.DataFrame(
            {
                "S1": indices["S1_conf"],
                "ST": indices["ST_conf"],
            },
            index=problem["names"],
        )

        si_df.plot.bar(
            yerr=err_df.values.T,
            ax=ax,
            capsize=4,
            width=0.75,
        )

        ax.set_title(policy_name.replace("_", " "))
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=45)
        ax.set_ylabel("Sobol index")

    fig.suptitle(
        f"First-order (S1) and total-order (ST) Sobol indices — {outcome}",
        fontsize=12,
    )

    plt.tight_layout()

    outpath = os.path.join(
        GSA_PLOTS_DIR,
        f"sobol_s1_st_{outcome}.png",
    )

    plt.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", outpath)

In [ ]:
# ============================================================
# SOBOL TOTAL-ORDER HEATMAP
# ============================================================

for policy_name in GSA_POLICY_NAMES:
    st_table = pd.DataFrame(
        index=problem["names"],
        columns=GSA_OBJECTIVES,
        dtype=float,
    )

    s1_table = pd.DataFrame(
        index=problem["names"],
        columns=GSA_OBJECTIVES,
        dtype=float,
    )

    for outcome in GSA_OBJECTIVES:
        indices = sobol_results[policy_name][outcome]

        if indices is None:
            continue

        st_table[outcome] = indices["ST"]
        s1_table[outcome] = indices["S1"]

    st_table = st_table.reindex(GSA_PARAMS)

    fig, ax = plt.subplots(figsize=(13, 4.8))

    im = ax.imshow(
        st_table.values,
        aspect="auto",
        vmin=0,
    )

    short_labels = [
        "welfare",
        "yrs > 2C",
        "frac > 2C",
        "temp 2100",
        "max temp",
        "wl damage",
        "wl abatement",
        "SA burden",
        "SA damage",
        "SA net output",
    ]

    ax.set_xticks(np.arange(len(GSA_OBJECTIVES)))
    ax.set_xticklabels(
        short_labels,
        rotation=35,
        ha="right",
    )

    ax.set_yticks(np.arange(len(GSA_PARAMS)))
    ax.set_yticklabels(GSA_PARAMS)

    for i in range(len(GSA_PARAMS)):
        for j in range(len(GSA_OBJECTIVES)):
            value = st_table.values[i, j]

            label = "" if not np.isfinite(value) else f"{value:.2f}"

            ax.text(
                j,
                i,
                label,
                ha="center",
                va="center",
                fontsize=8,
            )

    ax.set_title(
        f"Sobol total-order indices — {policy_name.replace('_', ' ')}"
    )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Total-order Sobol index ST")

    plt.tight_layout()

    outpath = os.path.join(
        GSA_PLOTS_DIR,
        f"sobol_total_order_heatmap_{policy_name}.png",
    )

    plt.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", outpath)

In [ ]:
# ============================================================
# SOBOL CONVERGENCE CHECK
# ============================================================

CONV_POLICY = "moderate_abatement"
CONV_OUTCOME = "zaf_mean_damage_fraction"

mask = experiments_sobol["policy"] == CONV_POLICY
Y_full = outcomes_sobol_df.loc[mask, CONV_OUTCOME].values.astype(float)

n_total = len(Y_full)
block = 2 * problem["num_vars"] + 2

step_size = max(block, (n_total // 20 // block) * block)

sample_sizes = [
    n for n in range(step_size, n_total + 1, step_size)
    if n % block == 0
]

convergence_data = pd.DataFrame(
    index=problem["names"],
    columns=sample_sizes,
    dtype=float,
)

for n in sample_sizes:
    indices = sobol_analyze.analyze(
        problem,
        Y_full[:n],
        calc_second_order=True,
        print_to_console=False,
    )

    convergence_data[n] = indices["ST"]

fig, ax = plt.subplots(figsize=(8, 4))

convergence_data.T.plot(ax=ax)

ax.set_xlabel("Sample size")
ax.set_ylabel("Total-order index ST")
ax.set_title(
    f"Sobol ST convergence — {CONV_POLICY}, {CONV_OUTCOME}"
)

ax.legend(bbox_to_anchor=(1, 1))

plt.tight_layout()

outpath = os.path.join(
    GSA_PLOTS_DIR,
    f"sobol_convergence_{CONV_POLICY}_{CONV_OUTCOME}.png",
)

plt.savefig(outpath, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", outpath)

In [ ]:
# ============================================================
# EXTRA-TREES FEATURE SCORING — COMPARISON WITH SOBOL
# ============================================================

et_scores = {}

for policy_name in GSA_POLICY_NAMES:
    mask = experiments_sobol["policy"] == policy_name

    x = experiments_sobol.loc[mask, GSA_PARAMS].copy()

    y = {
        outcome: outcomes_sobol_df.loc[mask, outcome].values
        for outcome in GSA_OBJECTIVES
    }

    scores = feature_scoring.get_feature_scores_all(x, y)
    et_scores[policy_name] = scores

    print(f"\nExtra-Trees feature importance — {policy_name}")
    display(scores.round(3))

In [ ]:
# ============================================================
# EXTRA-TREES HEATMAP
# ============================================================

for policy_name in GSA_POLICY_NAMES:
    scores = et_scores[policy_name].reindex(GSA_PARAMS)[GSA_OBJECTIVES]

    scores_norm = scores / scores.sum(axis=0)
    scores_norm = scores_norm.fillna(0)

    fig, ax = plt.subplots(figsize=(13, 4.8))

    im = ax.imshow(
        scores_norm.values,
        aspect="auto",
        vmin=0,
        vmax=1,
    )

    short_labels = [
        "welfare",
        "yrs > 2C",
        "frac > 2C",
        "temp 2100",
        "max temp",
        "wl damage",
        "wl abatement",
        "SA burden",
        "SA damage",
        "SA net output",
    ]

    ax.set_xticks(np.arange(len(GSA_OBJECTIVES)))
    ax.set_xticklabels(
        short_labels,
        rotation=35,
        ha="right",
    )

    ax.set_yticks(np.arange(len(GSA_PARAMS)))
    ax.set_yticklabels(GSA_PARAMS)

    for i in range(len(GSA_PARAMS)):
        for j in range(len(GSA_OBJECTIVES)):
            ax.text(
                j,
                i,
                f"{scores_norm.values[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8,
            )

    ax.set_title(
        f"Extra-Trees feature importance — {policy_name.replace('_', ' ')}"
    )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Normalised feature importance")

    plt.tight_layout()

    outpath = os.path.join(
        GSA_PLOTS_DIR,
        f"extratrees_heatmap_{policy_name}.png",
    )

    plt.savefig(outpath, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", outpath)

## 2. Optimisation setup

This section defines the adaptive RBF policy structure, welfare-function choice, reference SSP-RCP scenario, climate ensemble subset, objectives, epsilons, and configuration file.


### 2.1 Adaptive RBF policy design

In [ ]:
#  RBF decision variable structure.

n_rbfs = N_RBFS # dit is dus al ingevuld 
n_inputs = N_INPUTS
n_outputs = N_REGIONS
 

n_centers = n_rbfs * n_inputs
n_radii = n_rbfs * n_inputs
n_weights = n_rbfs * n_outputs
n_total = n_centers + n_radii + n_weights

rbf_structure = pd.DataFrame([
    {"Component": "Centers", "Formula": "n_rbfs * n_inputs", "Count": n_centers},
    {"Component": "Radii", "Formula": "n_rbfs * n_inputs", "Count": n_radii},
    {"Component": "Weights", "Formula": "n_rbfs * n_outputs", "Count": n_weights},
    {"Component": "Total", "Formula": "centers + radii + weights", "Count": n_total},
])

rbf_structure



> The RBF structure uses 244 decision variables. These variables encode an adaptive decision rule rather than a fixed mitigation action for every region and every year. This is politically relevant for South Africa because it avoids assuming that all countries must follow the same rigid mitigation pathway regardless of climate response or transition burden.

### 2.2 Welfare function choice

In [ ]:
WELFARE_OPTIONS = {
    "Utilitarian": WelfareFunction.UTILITARIAN,
    "Prioritarian": WelfareFunction.PRIORITARIAN,
    "Egalitarian": WelfareFunction.EGALITARIAN,
}

print("Welfare functions considered:")
for name, wf in WELFARE_OPTIONS.items():
    print(f"  {name:<20s} internal index = {wf.value[0]}")

CHOSEN_WELFARE_FUNCTION = WELFARE_OPTIONS["Prioritarian"]
CHOSEN_WELFARE_FUNCTION_IDX = CHOSEN_WELFARE_FUNCTION.value[0]

print(f"\nChosen: {CHOSEN_WELFARE_FUNCTION.name} (index {CHOSEN_WELFARE_FUNCTION_IDX})")
print("Rationale: Prioritarian welfare fits South Africa because it gives extra weight to worse-off regions.")

### 2.3 Reference SSP-RCP scenario

In [ ]:
CHOSEN_SCENARIO_INDEX = 2
CHOSEN_SCENARIO_NAME = "SSP2-RCP4.5"

print(f"Chosen reference scenario: {CHOSEN_SCENARIO_INDEX} — {CHOSEN_SCENARIO_NAME}")

### 2.4 FaIR ensemble strategy

In [ ]:
FULL_ENSEMBLE_SIZE = 1001
N_ENSEMBLE_LOCAL = 15

rng = np.random.default_rng(seed=42)
ensemble_indices = sorted(
    rng.choice(FULL_ENSEMBLE_SIZE, size=N_ENSEMBLE_LOCAL, replace=False).tolist()
)

speedup = FULL_ENSEMBLE_SIZE / N_ENSEMBLE_LOCAL

print("Selected ensemble indices:", ensemble_indices)
print(f"Approximate speedup vs full ensemble: {speedup:.1f}x")

### 2.5 Objectives and epsilon values

In [ ]:
from ema_workbench import ScalarOutcome

EPSILONS = [
    1.0,    # welfare
    0.01,   # fraction_above_threshold
    10.0,   # welfare_loss_damage
    10.0,   # welfare_loss_abatement
    0.001,  # abatement_burden

]

objectives =[
    ScalarOutcome("welfare", kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("fraction_above_threshold", kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("welfare_loss_damage", kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome("welfare_loss_abatement", kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome("abatement_burden", kind=ScalarOutcome.MINIMIZE),
]

objective_table = pd.DataFrame([
    {
        "Objective": obj.name,
        "Direction": "MINIMIZE" if obj.kind == ScalarOutcome.MINIMIZE else "MAXIMIZE",
        "Epsilon": eps,
        "South Africa interpretation": interp,
    }
    for obj, eps, interp in zip(
        objectives,
        EPSILONS,
        [
            "Lower welfare loss is better.",
            "Lower probability/fraction above 2°C is better.",
            "Assignment convention: larger stored value means less actual damage burden.",
            "Assignment convention: larger stored value means less actual abatement burden.",
            "Lower South Africa abatement cost as share of gross output is better.",
        ],
    )
])

objective_table



### 2.6 Save configuration



In [ ]:
student_config = {
    "start_year": START_YEAR,
    "end_year": END_YEAR,
    "data_timestep": DATA_TIMESTEP,
    "timestep": TIMESTEP,
    "emission_control_start_year": 2025,
    "n_rbfs": N_RBFS,
    "n_inputs": N_INPUTS,
    "epsilons": EPSILONS,
    "temperature_year_of_interest": 2100,
    "reference_ssp_rcp_scenario_index": CHOSEN_SCENARIO_INDEX,

    # These extra keys are useful for newer/local scripts. If a script ignores them, that is fine.
    "stochastic_run": False,
    "climate_ensemble_members": ensemble_indices,
    "social_welfare_function_type": CHOSEN_WELFARE_FUNCTION_IDX,
}

assert student_config["n_rbfs"] == student_config["n_inputs"] + 2
assert len(student_config["epsilons"]) == len(objectives)
assert all(e > 0 for e in student_config["epsilons"])
assert student_config["emission_control_start_year"] >= student_config["start_year"]

config_path = PROJECT_ROOT/'config'/ "config_student.json"
with open(config_path, "w") as fh:
    json.dump(student_config, fh, indent=4)

print(f"Config saved → {config_path}")
print(json.dumps(student_config, indent=4))

## 3. Generate candidate policies through MOEA

This section creates the optimisation command and loads completed Pareto-front files. The optimisation command itself is printed so it can be run in the correct environment.


### 3.1 Local optimisation run

Start with a small test run to ensure everything works. Increase `NFE` later if time allows.

- `NFE` = number of candidate policies tested per seed.
- `SEEDS` = independent optimiser runs.
- `N_ENSEMBLES` = climate ensemble members used during local optimisation.

In [ ]:
with open(config_path) as fh:
    cfg = json.load(fh)

explanations = {
    "start_year":                       "Simulation start year",
    "end_year":                         "Simulation end year",
    "data_timestep":                    "Years between raw input data points",
    "timestep":                         "Model integration timestep (years)",
    "emission_control_start_year":      "First year ECR can exceed zero",
    "n_rbfs":                           "Number of RBFs (effective: n_inputs + 2)",
    "n_inputs":                         "RBF input signals ",
    "epsilons":                         "Archive granularity",
    "temperature_year_of_interest":     "Year for threshold fraction evaluation",
    "reference_ssp_rcp_scenario_index": "Reference scenario index",
}

print(f"Configuration: {config_path}\n")
for k, v in cfg.items():
    print(f"  {k:<40s}  {str(v):<15}  # {explanations.get(k, '')}")

In [ ]:
DO NOT RUN, UNLESS YOU HAVE TIME 

NFE = 100000
SEEDS = [1, 2, 3, 4, 5]
N_ENSEMBLES = N_ENSEMBLE_LOCAL
N_PROCESSES = None
OUTPUT_DIR = RESULTS_DIR

seeds_str = " ".join(str(s) for s in SEEDS)
n_proc_arg = f"--n_processes {N_PROCESSES}" if N_PROCESSES else ""

script_path = (_PROJECT_DIR / "run_optimization_local.py").resolve()

cmd = (
    f"python {shlex.quote(str(script_path))} "
    f"--nfe {NFE} "
    f"--seeds {seeds_str} "
    f"--n_ensembles {N_ENSEMBLES} "
    f"--output_dir {shlex.quote(str(OUTPUT_DIR))} "
    f"--config {shlex.quote(str(config_path))} "
    + n_proc_arg
)

print("Command to run:")
print(cmd)

ret = os.system(cmd)
print(f"\nExit code: {ret} ({'OK' if ret == 0 else 'ERROR'})")

---
Each completed seed produces a directory e.g.  `<welfare_function>_<nfe>_<seed>/` inside `results/` containing:
- `pareto_front_<seed>.csv` — the final Pareto-optimal solutions (levers + objectives).
- `<welfare_function>_<nfe>_<seed>.tar.gz` — the ArchiveLogger convergence history (used in Assignment 6).
- `convergence_<seed>.csv` — EpsilonProgress and operator probabilities per NFE checkpoint (used in Assignment 6).

Load all available Pareto front CSVs and print a statistical summary.
>Tip: you can use print(all_results[OBJECTIVE_COLS].describe().round(n))
Print the nfes, and number of solutions in the pareto front as well.

In [ ]:

OBJECTIVE_COLS = [
    "welfare",
    "fraction_above_threshold",
    "welfare_loss_damage",
    "welfare_loss_abatement",
    "abatement_burden",
]

# Find all Pareto front CSV files in the results directory
pareto_files = sorted(Path(RESULTS_DIR).rglob("pareto_front_*.csv"))

print(f"Results directory: {RESULTS_DIR}")
print(f"Found {len(pareto_files)} Pareto front file(s).")

if len(pareto_files) == 0:
    raise FileNotFoundError(
        f"No pareto_front_*.csv files found in {RESULTS_DIR}. "
        "Check whether the optimisation wrote results to the correct folder."
    )

all_dfs = []

for file in pareto_files:
    df = pd.read_csv(file)

    # Folder name should look like PRIORITARIAN_500_9845531
    folder_name = file.parent.name
    parts = folder_name.split("_")

    welfare_function = parts[0] if len(parts) >= 1 else "unknown"
    nfe = int(parts[1]) if len(parts) >= 2 and parts[1].isdigit() else np.nan

    # File name should look like pareto_front_9845531.csv
    seed_match = re.search(r"pareto_front_(\d+)\.csv", file.name)
    seed = int(seed_match.group(1)) if seed_match else np.nan

    df["welfare_function"] = welfare_function
    df["nfe"] = nfe
    df["seed"] = seed
    df["source_file"] = str(file)

    all_dfs.append(df)

all_results = pd.concat(all_dfs, ignore_index=True)

# Optional: keep only the current South Africa welfare-function runs
# Uncomment if old UTILITARIAN runs are still in the results folder.
# all_results = all_results[all_results["welfare_function"] == "PRIORITARIAN"].copy()

# Check that all objective columns are available
missing_cols = [col for col in OBJECTIVE_COLS if col not in all_results.columns]
if missing_cols:
    raise KeyError(f"Missing objective columns in loaded Pareto fronts: {missing_cols}")

# Print nfes and number of solutions in each Pareto front
run_summary = (
    all_results
    .groupby(["welfare_function", "nfe", "seed"], dropna=False)
    .size()
    .reset_index(name="n_pareto_solutions")
    .sort_values(["welfare_function", "nfe", "seed"])
)

print("\nNFE and number of Pareto solutions per completed seed:")
display(run_summary)

print("\nStatistical summary of objective columns across all loaded Pareto solutions:")
display(all_results[OBJECTIVE_COLS].describe().round(3))

In [ ]:
variation = all_results[OBJECTIVE_COLS].nunique(dropna=True)

has_more_than_one_solution = len(all_results) > 1
has_variation_all_objectives = (variation > 1).all()
has_some_fraction_below_1 = (all_results["fraction_above_threshold"] < 1.0).any()

non_trivial_checks = pd.DataFrame({
    "Check": [
        "More than 1 solution",
        "Variation in all four objective columns",
        "At least one solution has fraction_above_threshold < 1.0",
    ],
    "Result": [
        has_more_than_one_solution,
        has_variation_all_objectives,
        has_some_fraction_below_1,
    ],
})

display(non_trivial_checks)

print("\nVariation per objective column:")
display(variation.to_frame("n_unique_values"))

if not has_more_than_one_solution:
    print("Warning: Only one solution loaded. This is not enough for a meaningful Pareto front.")

if not has_variation_all_objectives:
    print("Warning: At least one objective has no variation. This may be due to very low NFE or a setup issue.")

if not has_some_fraction_below_1:
    print("Warning: All solutions have fraction_above_threshold = 1.0. The run may not have found climate-effective policies yet.")

## 4. Check search quality and build a reference set

This section merges seed results, removes dominated alternatives, checks convergence information, and saves a canonical reference set.


In [ ]:
import glob

csv_paths = sorted(glob.glob(
    os.path.join(RESULTS_DIR, "**", "pareto_front_*.csv"), recursive=True
))

if not csv_paths:
    raise FileNotFoundError(
        f"No pareto_front_*.csv found in {RESULTS_DIR}.\n"
        "Run Assignment 5 (run_optimization_local.py) first."
    )

# nfe_groups[nfe][seed] = DataFrame
nfe_groups = {}

for path in csv_paths:
    dir_name = os.path.basename(os.path.dirname(path))
    parts    = dir_name.split("_")

    try:
        nfe  = int(parts[-2])
        seed = int(parts[-1])
    except (ValueError, IndexError):
        seed = int(os.path.basename(path).replace("pareto_front_", "").replace(".csv", ""))
        nfe  = 0

    df = pd.read_csv(path)

    # Keep only valid/non-penalty policies
    df = df[df["welfare"] < 1e5].reset_index(drop=True)

    # Prevent zero radii/weights from old saved Pareto-front policies
    radii_cols = [c for c in df.columns if c.startswith("radii")]
    weights_cols = [c for c in df.columns if c.startswith("weights")]

    if radii_cols:
        df[radii_cols] = df[radii_cols].clip(lower=SMALL_NUMBER)

    if weights_cols:
        df[weights_cols] = df[weights_cols].clip(lower=SMALL_NUMBER)

    nfe_groups.setdefault(nfe, {})[seed] = df
# ── Discover convergence archives — keyed by (nfe, seed) ─────────────────────
archive_paths = sorted(glob.glob(
    os.path.join(RESULTS_DIR, "**", "PRIORITARIAN_*.tar.gz"), recursive=True
))

nfe_seed_archives = {}   # (nfe, seed) → path
for p in archive_paths:
    parts    = os.path.basename(p).replace(".tar.gz", "").split("_")
    seed_val = int(parts[-1])
    nfe_val  = int(parts[-2])
    nfe_seed_archives[(nfe_val, seed_val)] = p

# ── Discover convergence CSVs — keyed by (nfe, seed) ─────────────────────────
# Saved by run_optimization_local.py as results/<welfare_function>_<nfe>_<seed>/convergence_<seed>.csv
conv_csv_paths = sorted(glob.glob(
    os.path.join(RESULTS_DIR, "**", "convergence_*.csv"), recursive=True
))

nfe_seed_convergence = {}   # (nfe, seed) → DataFrame
for p in conv_csv_paths:
    dir_name = os.path.basename(os.path.dirname(p))
    parts    = dir_name.split("_")
    try:
        nfe_val  = int(parts[-2])
        seed_val = int(parts[-1])
    except (ValueError, IndexError):
        seed_val = int(os.path.basename(p).replace("convergence_", "").replace(".csv", ""))
        nfe_val  = 0
    df_conv = pd.read_csv(p, index_col=0)
    nfe_seed_convergence[(nfe_val, seed_val)] = df_conv

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"NFE groups found: {sorted(nfe_groups)}")
print()
rows = []
for nfe in sorted(nfe_groups):
    for seed, df in sorted(nfe_groups[nfe].items()):
        rows.append({
            "nfe_budget":      nfe,
            "seed":            seed,
            "n_solutions":     len(df),
            "has_archive":     (nfe, seed) in nfe_seed_archives,
            "has_conv_csv":    (nfe, seed) in nfe_seed_convergence,
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print(f"\n{len(nfe_seed_archives)} archive(s) | {len(nfe_seed_convergence)} convergence CSV(s)")

In [ ]:
import matplotlib.path as _mpath

def _fixed_path_deepcopy(self, memo):
    cls   = type(self)
    verts = copy.deepcopy(self.vertices, memo)
    codes = copy.deepcopy(self.codes, memo) if self.codes is not None else None
    new   = cls.__new__(cls)
    new.__init__(verts, codes)
    return new

_mpath.Path.__deepcopy__ = _fixed_path_deepcopy

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# EMA Workbench -- core optimization
from ema_workbench.em_framework.optimization import (
    epsilon_nondominated,
    PlatypusProblem,
    load_archives,
)
# EMA Workbench -- native convergence metrics (3.0+)
from ema_workbench.em_framework.optimization_convergence import (
    HypervolumeMetric,
    GenerationalDistanceMetric,
    EpsilonIndicatorMetric,
)
from platypus import Real

# Objective metadata
OBJECTIVE_COLS  = ["welfare", "fraction_above_threshold",
                   "welfare_loss_damage", "welfare_loss_abatement", "abatement_burden"]
MAXIMIZE_COLS   = ["welfare_loss_damage", "welfare_loss_abatement"]
MINIMIZE_COLS   = ["welfare", "fraction_above_threshold", "abatement_burden"]
OBJECTIVE_LABELS = {
    "welfare":                  "Welfare loss\n(MINIMIZE)",
    "fraction_above_threshold": "Fraction above\n2 C in 2100\n(MINIMIZE)",
    "welfare_loss_damage":      "Welfare loss\nfrom damage\n(MAXIMIZE)",
    "welfare_loss_abatement":   "Welfare loss\nfrom abatement\n(MAXIMIZE)",
    "abatement_burden": "Abt\nabatement burden\n(MINIMIZE)"
}

print(f"JUSTICE root : {_JUSTICE_ROOT}")
print(f"Results root : {RESULTS_DIR}")
print("EMA Workbench native convergence metrics : OK")

In [ ]:

_sample_df = next(df for nfe in nfe_groups for df in nfe_groups[nfe].values())

# Only RBF decision variables are levers
_lever_cols = [
    c for c in _sample_df.columns
    if c.startswith("center ") or c.startswith("radii ") or c.startswith("weights ")
]

_lever_cols_s = [c.replace(" ", "_") for c in _lever_cols]
_n_vars = len(_lever_cols)

print(f"Detected {_n_vars} lever columns.")
problem = PlatypusProblem(_n_vars, len(OBJECTIVE_COLS))

problem.types = (
    [Real(-1.0, 1.0)] * 8
    + [Real(0.0, 1.0)] * 8
    + [Real(0.0, 1.0)] * 228
)

problem.directions = [
    PlatypusProblem.MINIMIZE,
    PlatypusProblem.MINIMIZE,
    PlatypusProblem.MAXIMIZE,
    PlatypusProblem.MAXIMIZE,
    PlatypusProblem.MINIMIZE, #ABTAEMENT_BURDEN
]


problem.parameter_names = _lever_cols_s
problem.outcome_names = OBJECTIVE_COLS

# Compatibility patch for this EMA Workbench version:
# Sample._from_platypus_solution expects decision_variables with .name and .shape.
class _DummyDecisionVariable:
    def __init__(self, name):
        self.name = name
        self.shape = None

problem.decision_variables = [
    _DummyDecisionVariable(name) for name in _lever_cols_s
]

# Use the same epsilons as in your optimisation config
EPSILONS = [1.0, 0.01, 10.0, 10.0, 0.001]

def sanitize_cols(df):
    return df.rename(columns=lambda c: c.replace(" ", "_"))

def restore_cols(df):
    return df.rename(
        columns=lambda c: c.replace("_", " ", 1)
        if c.startswith(("center_", "radii_", "weights_"))
        else c
    )

# Build one reference set per NFE group
nfe_ref_sets = {}

for nfe in sorted(nfe_groups):
    seed_dict = nfe_groups[nfe]

    dfs_san = []
    for seed, df in seed_dict.items():
        keep_cols = _lever_cols + OBJECTIVE_COLS
        df_clean = df[keep_cols].dropna(subset=OBJECTIVE_COLS).copy()

        # Remove penalty / failed solutions
        df_clean = df_clean[df_clean["welfare"] < 1e5].copy()

        dfs_san.append(sanitize_cols(df_clean))

    n_in = sum(len(d) for d in dfs_san)
    print(f"\nNFE={nfe:,}: merging {n_in} solutions from {len(seed_dict)} seed(s)...")

    ref_s = epsilon_nondominated(dfs_san, EPSILONS, problem)
    ref = restore_cols(ref_s)

    nfe_ref_sets[nfe] = ref

    pruning = (1 - len(ref) / n_in) * 100 if n_in > 0 else 0
    print(f"  Reference set: {len(ref)} solutions (pruned {pruning:.0f}%)")

    print("  Objective ranges:")
    for col in OBJECTIVE_COLS:
        print(f"    {col:<40s} {ref[col].min():.3f} -- {ref[col].max():.3f}")

    ref_path = os.path.join(str(RESULTS_DIR), f"reference_set_prioritarian_{nfe}.csv")
    ref.to_csv(ref_path, index=False)
    print(f"  Saved -> {ref_path}")

# Save the highest-NFE group as the canonical reference set
best_nfe = max(nfe_ref_sets)

canonical_path = os.path.join(str(RESULTS_DIR), "reference_set_prioritarian.csv")
nfe_ref_sets[best_nfe].to_csv(canonical_path, index=False)

print(f"\nCanonical reference set (NFE={best_nfe:,}) -> {canonical_path}")

### 4.1 Load outputs



In [ ]:

REFERENCE_SET_PATH = (RESULTS_DIR / "reference_set_prioritarian_100000.csv")
print(REFERENCE_SET_PATH)

In [ ]:
archive_paths = sorted(glob.glob(
    os.path.join(str(RESULTS_DIR), "**", "*.tar.gz"),
    recursive=True
))

nfe_seed_archives = {}

for p in archive_paths:
    filename = os.path.basename(p).replace(".tar.gz", "")
    parts = filename.split("_")

    try:
        welfare_function = "_".join(parts[:-2])
        nfe_val = int(parts[-2])
        seed_val = int(parts[-1])
    except (ValueError, IndexError):
        continue

    if welfare_function == "PRIORITARIAN":
        nfe_seed_archives[(nfe_val, seed_val)] = p

print(f"Found {len(nfe_seed_archives)} PRIORITARIAN archive(s).")

for (nfe, seed), path in sorted(nfe_seed_archives.items()):
    print(f"NFE={nfe}, seed={seed}: {path}")

In [ ]:


if not nfe_seed_archives:
    print("No convergence archives found -- skipping metric computation.")
else:
    # all_snapshots[(nfe, seed)] = {checkpoint_nfe: DataFrame}
    all_snapshots = {}

    for (nfe, seed), path in sorted(nfe_seed_archives.items()):
        print(f"  Loading: NFE={nfe:,}  seed={seed} ...", end=" ")
        # load_archives returns list[tuple[int, DataFrame]] in ema_workbench 3.0
        snaps = dict(load_archives(path))
        snaps = {
            n: df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")],
                       errors="ignore")
            for n, df in snaps.items()
        }
        all_snapshots[(nfe, seed)] = snaps
        max_n = max(snaps)
        print(f"{len(snaps)} snapshots | final archive = {len(snaps[max_n])} solutions @ NFE {max_n:,}")

    print(f"\n{len(all_snapshots)} archive(s) loaded.")


In [ ]:
def sanitize_cols(df):
    return df.rename(columns=lambda c: c.replace(" ", "_"))


def _build_metrics(ref_df, problem):
    """Create the three metric objects for one NFE group.
    ref_df must have sanitized column names."""
    hv_m  = HypervolumeMetric(ref_df, problem)
    gd_m  = GenerationalDistanceMetric(ref_df, problem)
    ei_m  = EpsilonIndicatorMetric(ref_df, problem)
    return hv_m, gd_m, ei_m

reference_set = "reference_set_prioritarian_100000.csv"

nfe_metrics = {}  # nfe -> (hv_metric, gd_metric, ei_metric)
for nfe, ref in nfe_ref_sets.items():
    ref_san = sanitize_cols(ref)
    nfe_metrics[nfe] = _build_metrics(ref_san, problem)
print("Native convergence metrics built for NFE groups:", sorted(nfe_metrics))


In [ ]:
if "all_snapshots" in dir() and "nfe_ref_sets" in dir():
    all_metric_curves = {}

    for nfe_budget in sorted(nfe_groups):
        hv_m, gd_m, ei_m = nfe_metrics[nfe_budget]

        group_snapshots = {
            seed: snaps
            for (nfe, seed), snaps in all_snapshots.items()
            if nfe == nfe_budget
        }

        if not group_snapshots:
            print(f"NFE={nfe_budget:,}: no archive snapshots -- skipping metrics.")
            continue

        print(f"\nNFE budget = {nfe_budget:,}  |  {len(group_snapshots)} seed(s)")

        for seed, snaps in sorted(group_snapshots.items()):
            nfes = sorted(snaps.keys())
            nfes = nfes[::10] + [nfes[-1]]
            nfes = sorted(set(nfes))

            sizes = [len(snaps[n]) for n in nfes]

            hvs, gds, eis = [], [], []

            print(f"  seed {seed}: calculating {len(nfes)} snapshots...")

            for i, n in enumerate(nfes, start=1):
                snap_san = sanitize_cols(
                    snaps[n][snaps[n]["welfare"] < 1e5]
                    if "welfare" in snaps[n].columns else snaps[n]
                )

                print(
                    f"    snapshot {i}/{len(nfes)} | "
                    f"NFE={n:,} | archive size={len(snap_san)}",
                    flush=True
                )

                if n == nfes[-1]:
                    try:
                        hvs.append(hv_m.calculate(snap_san))
                    except Exception:
                        hvs.append(float("nan"))
                else:
                    hvs.append(float("nan"))

                try:
                    gds.append(gd_m.calculate(snap_san))
                except Exception as e:
                    print(f"      GD failed: {e}")
                    gds.append(float("nan"))

                try:
                    eis.append(ei_m.calculate(snap_san))
                except Exception as e:
                    print(f"      EI failed: {e}")
                    eis.append(float("nan"))

            hvs = np.array(hvs)
            gds = np.array(gds)
            eis = np.array(eis)

            eps_source = "proxy"
            eps = np.diff(sizes, prepend=sizes[0])

            if (nfe_budget, seed) in nfe_seed_convergence:
                _conv = nfe_seed_convergence[(nfe_budget, seed)]
                if "epsilon_progress" in _conv.columns and "nfe" in _conv.columns:
                    eps = np.round(
                        np.interp(
                            np.array(nfes),
                            _conv["nfe"].values,
                            _conv["epsilon_progress"].values
                        )
                    ).astype(int)
                    eps_source = "EMA Workbench"

            all_metric_curves[(nfe_budget, seed)] = {
                "nfe":        np.array(nfes),
                "hv":         hvs,
                "eps":        eps,
                "gd":         gds,
                "ei":         eis,
                "eps_source": eps_source,
            }

            print(
                f"  seed {seed}: HV={hvs[-1]:.5f} | GD={gds[-1]:.5f} | "
                f"EI={eis[-1]:.5f} | eps_source={eps_source}"
            )

    print(f"\nDone -- {len(all_metric_curves)} seed curve(s) computed.")

In [ ]:
# Four-panel convergence plot -- one figure per NFE group
if "all_metric_curves" in dir():

    nfe_budgets = sorted({nfe for (nfe, _) in all_metric_curves})
    _tab10 = plt.cm.tab10

    for nfe_budget in nfe_budgets:
        keys   = [(nfe, seed) for (nfe, seed) in sorted(all_metric_curves)
                  if nfe == nfe_budget]
        seeds  = [seed for (_, seed) in keys]
        colors = [_tab10(i / 10) for i in range(len(seeds))]

        fig, axes = plt.subplots(4, 1, figsize=(11, 16), sharex=True)

        # Panel 1: Hypervolume (DEAP)
        ax = axes[0]
        for (nfe, seed), col in zip(keys, colors):
            d = all_metric_curves[(nfe, seed)]
            ax.plot(d["nfe"], d["hv"], lw=2.2, color=col, marker="o", markersize=4,
                    markevery=max(1, len(d["nfe"]) // 15), label=f"seed {seed}")
        ax.set_ylabel("Normalised Hypervolume", fontsize=11)
        ax.set_title("Hypervolume (DEAP)   higher = better\nPlateau = convergence", fontsize=10)
        ax.legend(fontsize=9, ncol=min(len(seeds), 3))

        # Panel 2: Epsilon-progress
        ax = axes[1]
        for (nfe, seed), col in zip(keys, colors):
            d = all_metric_curves[(nfe, seed)]
            ax.plot(d["nfe"], d["eps"], lw=2.0, color=col,
                    drawstyle="steps-post", label=f"seed {seed}")
            ax.fill_between(d["nfe"], 0, d["eps"], step="post", color=col, alpha=0.08)
        ax.axhline(0, color="0.5", lw=0.8, linestyle="--")
        ax.set_ylabel("Archive size change", fontsize=11)
        ax.set_title("Epsilon-Progress   positive = new solutions added\nPlateau at 0 = convergence", fontsize=10)
        ax.legend(fontsize=9, ncol=min(len(seeds), 3))

        # Panel 3: Generational Distance
        ax = axes[2]
        for (nfe, seed), col in zip(keys, colors):
            d = all_metric_curves[(nfe, seed)]
            mask = np.isfinite(d["gd"])
            ax.plot(d["nfe"][mask], d["gd"][mask], lw=2.2, color=col,
                    marker="o", markersize=4,
                    markevery=max(1, mask.sum() // 15), label=f"seed {seed}")
        ax.set_ylabel("Generational Distance", fontsize=11)
        ax.set_title("Generational Distance (Platypus)   lower = archive closer to reference", fontsize=10)
        ax.legend(fontsize=9, ncol=min(len(seeds), 3))

        # Panel 4: Epsilon Indicator
        ax = axes[3]
        for (nfe, seed), col in zip(keys, colors):
            d = all_metric_curves[(nfe, seed)]
            mask = np.isfinite(d["ei"])
            ax.plot(d["nfe"][mask], d["ei"][mask], lw=2.2, color=col,
                    marker="s", markersize=4,
                    markevery=max(1, mask.sum() // 15), label=f"seed {seed}")
        ax.axhline(0, color="0.5", lw=0.8, linestyle="--")
        ax.set_ylabel("Epsilon Indicator", fontsize=11)
        ax.set_title("Epsilon Indicator (Platypus)   lower = result closer to dominating reference\nEI = 0 means result set fully epsilon-dominates the reference", fontsize=10)
        ax.legend(fontsize=9, ncol=min(len(seeds), 3))
        ax.set_xlabel("Number of Function Evaluations (NFE)", fontsize=11)

        for ax in axes:
            ax.grid(axis="y", color="0.9", zorder=0)

        fig.suptitle(
            f"MOEA Convergence Metrics -- NFE budget = {nfe_budget:,}\n"
            f"GenerationalBorg on JUSTICE (Utilitarian SWF)   |   {len(seeds)} seed(s)",
            fontsize=12, y=1.01, fontweight="bold",
        )
        plt.tight_layout()
        fname = f"moea_convergence_metrics_{nfe_budget}nfe.png"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved: {fname}")

In [ ]:
# Final metric summary table
if "all_metric_curves" in dir():
    rows = []
    for (nfe_budget, seed), d in sorted(all_metric_curves.items()):
        rows.append({
            "nfe_budget":       nfe_budget,
            "seed":             seed,
            "final_NFE":        int(d["nfe"][-1]),
            "n_snapshots":      len(d["nfe"]),
            "final_HV":         round(float(d["hv"][-1]), 5)  if np.isfinite(d["hv"][-1])  else None,
            "total_eps":        int(d["eps"].sum()),
            "final_GD":         round(float(d["gd"][-1]), 5)  if np.isfinite(d["gd"][-1])  else None,
            "final_EI":         round(float(d["ei"][-1]), 5)  if np.isfinite(d["ei"][-1])  else None,
            "eps_source":       d["eps_source"],
        })

    df_metrics = pd.DataFrame(rows)
    print("MOEA performance summary:")
    print(df_metrics.to_string(index=False))

    print("\n-- Per-NFE-group seed consistency (HV coefficient of variation) --")
    for nfe_budget, grp in df_metrics.groupby("nfe_budget"):
        hv_vals = grp["final_HV"].dropna()
        if len(hv_vals) > 1:
            cv = hv_vals.std() / hv_vals.mean() * 100
            print(f"  NFE={nfe_budget:,}: HV CV = {cv:.1f}%  "
                  f"({'high consistency' if cv < 5 else 'moderate' if cv < 20 else 'high variability'})")
        else:
            print(f"  NFE={nfe_budget:,}: only 1 seed -- no CV computable")

## 5. Understand trade-offs and select candidate policies

This section turns the Pareto set into interpretable evidence: pairwise plots, parallel coordinates, selected policy IDs, and ECR pathways.


### 5.1 Pareto trade-off plots

In [ ]:
import copy
# Step 5.1 — Pareto trade-off plots for optimisation objectives
reference_file = pd.read_csv(RESULTS_DIR / reference_set)
if reference_file is None:
    raise ValueError("No reference set loaded. Load reference_set_prioritarian.csv first.")

required = set(OBJECTIVE_COLS)
missing = required - set(reference_file.columns)

if missing:
    raise KeyError(f"Reference set is missing expected objective columns: {missing}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1. Climate risk vs abatement objective
axes[0].scatter(
    reference_file["fraction_above_threshold"],
    reference_file["welfare_loss_abatement"],
    alpha=0.7,
)
axes[0].set_xlabel("Fraction above temperature threshold")
axes[0].set_ylabel("Abatement-related welfare objective")
axes[0].set_title("Climate risk vs abatement objective")
axes[0].grid(True, alpha=0.3)

# 1. Climate risk vs  abatement burden
axes[1].scatter(
    reference_file["fraction_above_threshold"],
    reference_file["abatement_burden"],
    alpha=0.7,
)
axes[1].set_xlabel("Fraction above temperature threshold")
axes[1].set_ylabel("South Africa abatement burden")
axes[1].set_title("Climate risk vs abatement burden")
axes[1].grid(True, alpha=0.3)

# 2. Damage objective vs abatement objective
axes[2].scatter(
    reference_file["welfare_loss_damage"],
    reference_file["welfare_loss_abatement"],
    alpha=0.7,
)
axes[2].set_xlabel("Damage-related welfare objective")
axes[2].set_ylabel("Abatement-related welfare objective")
axes[2].set_title("Damage vs abatement objective")
axes[2].grid(True, alpha=0.3)

# 2. Abatement welfare objective vs South Africa burden
axes[3].scatter(
    reference_file["welfare_loss_abatement"],
    reference_file["abatement_burden"],
    alpha=0.7,
)
axes[3].set_xlabel("Abatement-related welfare objective")
axes[3].set_ylabel("abatement burden")
axes[3].set_title("Global abatement objective vs burden")
axes[3].grid(True, alpha=0.3)

# 3. Welfare vs climate risk
axes[4].scatter(
    reference_file["welfare"],
    reference_file["fraction_above_threshold"],
    alpha=0.7,
)
axes[4].set_xlabel("Welfare objective")
axes[4].set_ylabel("Fraction above temperature threshold")
axes[4].set_title("Welfare vs climate risk")
axes[4].grid(True, alpha=0.3)

plt.suptitle("Pareto trade-offs — Prioritarian reference set", fontsize=13)
plt.tight_layout()

plot_path = PLOTS_DIR / "pareto_tradeoffs_prioritarian.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Figure saved:", plot_path)

In [ ]:
# Step 5.2 — Parallel-coordinates plot for optimisation objectives
# All axes are normalised so that up = better.

obj = reference_file[OBJECTIVE_COLS].copy()

def normalise_for_plot(df_obj):
    normed = df_obj.copy().astype(float)

    # For MINIMIZE objectives: lower is better, so invert after scaling
    for col in MINIMIZE_COLS:
        lo, hi = normed[col].min(), normed[col].max()
        normed[col] = 1.0 - (normed[col] - lo) / (hi - lo + 1e-15)

    # For MAXIMIZE objectives: higher is better
    for col in MAXIMIZE_COLS:
        lo, hi = normed[col].min(), normed[col].max()
        normed[col] = (normed[col] - lo) / (hi - lo + 1e-15)

    return normed

obj_norm = normalise_for_plot(obj)

# Four highlighted anchor policies
extremes = {
    "Lowest climate risk": obj["fraction_above_threshold"].idxmin(),
    "Best welfare": obj["welfare"].idxmin(),
    "Best abatement objective": obj["welfare_loss_abatement"].idxmax(),
    "Best damage objective": obj["welfare_loss_damage"].idxmax(),
    "Lowest abatement burden": obj["abatement_burden"].idxmin(),
}

extreme_colors = ["#1565C0", "#B71C1C", "#2E7D32", "#E65100","#6A1B9A" ]

fig, ax = plt.subplots(figsize=(11, 6))

x_pos = np.arange(len(OBJECTIVE_COLS))
axis_labels = [OBJECTIVE_LABELS[c] for c in OBJECTIVE_COLS]

# Colour all lines by original climate risk
risk_vals = obj["fraction_above_threshold"].values
cmap_risk = plt.cm.RdYlBu_r
norm_risk = Normalize(vmin=risk_vals.min(), vmax=risk_vals.max())

# Plot all reference-set policies
for idx, row in obj_norm.iterrows():
    y = row[OBJECTIVE_COLS].values
    color = cmap_risk(norm_risk(obj.loc[idx, "fraction_above_threshold"]))
    ax.plot(x_pos, y, color=color, lw=1.2, alpha=0.45, zorder=2)

# Highlight anchor policies
handles = []

for (label, idx), ec in zip(extremes.items(), extreme_colors):
    h, = ax.plot(
        x_pos,
        obj_norm.loc[idx, OBJECTIVE_COLS].values,
        color=ec,
        lw=3.2,
        label=label,
        zorder=5,
    )
    handles.append(h)

# Axis styling
for xi in x_pos:
    ax.axvline(xi, color="0.75", lw=0.8, zorder=1)

ax.set_xticks(x_pos)
ax.set_xticklabels(axis_labels, fontsize=10)
ax.set_ylabel("Normalised performance (up = better)", fontsize=10)
ax.set_ylim(-0.08, 1.12)

ax.set_title(
    "Parallel-Coordinates Plot — Prioritarian Reference Set\n"
    "(all axes: up = better; colour = climate threshold risk)",
    fontsize=11,
)

ax.legend(handles=handles, loc="upper right", fontsize=9)

# Colourbar
sm = plt.cm.ScalarMappable(cmap=cmap_risk, norm=norm_risk)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label("Fraction above temperature threshold")

plt.tight_layout()

plot_path = PLOTS_DIR / "parcoords_reference_set_prioritarian.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Figure saved:", plot_path)

In [ ]:
# ── JUSTICE imports ───────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

from justice.model import JUSTICE
from justice.util.data_loader import DataLoader
from justice.util.enumerations import (
    Abatement, DamageFunction, Economy, WelfareFunction
)
from justice.util.emission_control_constraint import EmissionControlConstraint
from justice.util.model_time import TimeHorizon
from solvers.emodps.rbf import RBF

with open(os.path.join(_PROJECT_DIR, "config/config_student.json")) as fh:
    _cfg = json.load(fh)

_time_horizon = TimeHorizon(
    start_year            = _cfg["start_year"],
    end_year              = _cfg["end_year"],
    data_timestep         = _cfg["data_timestep"],
    timestep              = _cfg["timestep"],
)
N_TIMESTEPS    = len(_time_horizon.model_time_horizon)
N_REGIONS      = len(DataLoader().REGION_LIST)
REGION_LIST    = list(DataLoader().REGION_LIST)
N_INPUTS_RBF   = _cfg["n_inputs"]
N_RBFS         = _cfg["n_inputs"] + 2
SCENARIO       = _cfg["reference_ssp_rcp_scenario_index"]
EC_START_TS    = _time_horizon.year_to_timestep(
    year=_cfg["emission_control_start_year"],
    timestep=_cfg["timestep"],
)
_MAX_TEMP, _MIN_TEMP = 16.0, 0.0
_MAX_DIFF, _MIN_DIFF = 2.0,  0.0

# ── Helper: run JUSTICE for one policy row ────────────────────────────────────
def run_policy_ecr(policy_row, n_ensemble=1):
    rbf = RBF(n_rbfs=N_RBFS, n_inputs=N_INPUTS_RBF, n_outputs=N_REGIONS)
    c_shape, r_shape, w_shape = rbf.get_shape()

    centers = np.array([policy_row[f"center {i}"] for i in range(c_shape[0])])
    radii   = np.array([policy_row[f"radii {i}"]  for i in range(r_shape[0])])
    weights = np.array([policy_row[f"weights {i}"] for i in range(w_shape[0])])
    rbf.set_decision_vars(np.concatenate([centers, radii, weights]))

    constraint = EmissionControlConstraint(
        max_annual_growth_rate=0.04,
        emission_control_start_timestep=EC_START_TS,
        min_emission_control_rate=0.01,
    )

    ensemble_indices = list(np.linspace(1, 1000, max(n_ensemble, 10), dtype=int))[:n_ensemble]

    model = JUSTICE(
        scenario=SCENARIO,
        climate_ensembles=ensemble_indices,
        economy_type=Economy.NEOCLASSICAL,
        damage_function_type=DamageFunction.KALKUHL,
        abatement_type=Abatement.ENERDATA,
        social_welfare_function_type=WelfareFunction.PRIORITARIAN.value[0],
    )
    no_ens = model.no_of_ensembles

    ecr             = np.zeros((N_REGIONS, N_TIMESTEPS, no_ens))
    constrained_ecr = np.zeros_like(ecr)
    prev_temp = 0.0
    diff      = 0.0

    for t in range(N_TIMESTEPS):
        constrained_ecr[:, t, :] = constraint.constrain_emission_control_rate(
            ecr[:, t, :], t, allow_fallback=False
        )
        model.stepwise_run(
            emission_control_rate=constrained_ecr[:, t, :],
            timestep=t,
            endogenous_savings_rate=True,
        )
        data = model.stepwise_evaluate(timestep=t)
        temp = data["global_temperature"][t, :]

        if t % 5 == 0:
            diff      = temp - prev_temp
            prev_temp = temp

        scaled_temp = (temp - _MIN_TEMP) / (_MAX_TEMP - _MIN_TEMP)
        scaled_diff = (diff - _MIN_DIFF) / (_MAX_DIFF - _MIN_DIFF)

        if t < N_TIMESTEPS - 1:
            ecr[:, t + 1, :] = rbf.apply_rbfs(np.array([scaled_temp, scaled_diff]))

    datasets = model.evaluate()
    ecr_mean = constrained_ecr.mean(axis=2)
    return ecr_mean, datasets




In [ ]:
# Step 5.3 — Reconstruct ECR pathways for selected policies

# ── Select your policies ────────────────────────────────────
# Use the same policies you highlighted in your parallel-coordinates plot.

ref_set = reference_file.copy()
obj_ref = ref_set[OBJECTIVE_COLS]

idx_1 = obj_ref["fraction_above_threshold"].idxmin()
idx_2 = obj_ref["welfare"].idxmin()
idx_3 = obj_ref["welfare_loss_abatement"].idxmax()
idx_4 = obj_ref["welfare_loss_damage"].idxmax()
idx_5 = obj_ref["abatement_burden"].idxmin()
anchors = {
    "Policy 1 — lowest climate risk": idx_1,
    "Policy 2 — best welfare": idx_2,
    "Policy 3 — best abatement objective": idx_3,
    "Policy 4 — best damage objective": idx_4,
    "Policy 5 — lowest abatement burden": idx_5,
}

print("Selected policies from Pareto reference set:\n")

for label, idx in anchors.items():
    print(f"  {label}")
    print(obj_ref.loc[idx].round(3).to_string(index=True))
    print()

# ── Run JUSTICE for each anchor ───────────────────────────────────────────────

print("Running JUSTICE model for all selected policies ...")

ecr_1, data_1 = run_policy_ecr(ref_set.loc[idx_1], n_ensemble=15)
ecr_2, data_2 = run_policy_ecr(ref_set.loc[idx_2], n_ensemble=15)
ecr_3, data_3 = run_policy_ecr(ref_set.loc[idx_3], n_ensemble=15)
ecr_4, data_4 = run_policy_ecr(ref_set.loc[idx_4], n_ensemble=15)
ecr_5, data_5 = run_policy_ecr(ref_set.loc[idx_5], n_ensemble=15)

print("Done.")

## 6. ECR and burden maps for selected policies

This section reconstructs emission control rates for selected policies and maps the spatial distribution of mitigation effort and abatement burden. This is important for explaining what the optimised policies actually mean.


In [ ]:
# ── Map world countries to RICE50 regions via ISO-3166 codes ─────────────────
import importlib.util, pathlib, json as _json
import geopandas as gpd

_rice50_dict_path = os.path.join(_JUSTICE_ROOT, "data", "input", "rice50_regions_dict.json")
with open(_rice50_dict_path) as _f:
    _rice50_dict = _json.load(_f)

iso_to_rice50 = {
    iso: region
    for region, isos in _rice50_dict.items()
    for iso in isos
}

_name_fallback = {
    "France":     "fra",
    "Norway":     "nor",
    "Kosovo":     "oeu",
    "N. Cyprus":  "tur",
    "Somaliland": "rsaf",
}

# ── Extract end-of-horizon ECR snapshot ───────────────────────────────────────
t_end     = ecr_1.shape[1] - 1
snap_year = int(_time_horizon.model_time_horizon[t_end])
print(f"End-of-horizon snapshot: timestep {t_end} = year {snap_year}")

def _snap_end(arr):
    return {REGION_LIST[i]: arr[i, t_end] for i in range(N_REGIONS)}

# TODO: replace the keys with your own policy labels (must match what you used above)
ecr_end = {
    "Policy 1 — lowest climate risk": _snap_end(ecr_1),
    "Policy 2 — best welfare": _snap_end(ecr_2),
    "Policy 3 — best abatement objective": _snap_end(ecr_3),
    "Policy 4 — best damage objective": _snap_end(ecr_4),
    "Policy 5 — lowest abatement burden": _snap_end(ecr_5),
}

# ── Load world shapefile and assign regions ───────────────────────────────────
_pyogrio_path = pathlib.Path(importlib.util.find_spec("pyogrio").origin).parent
_ne_shp = str(_pyogrio_path / "tests" / "fixtures" / "naturalearth_lowres" / "naturalearth_lowres.shp")
world = gpd.read_file(_ne_shp)

world["rice50"] = world["iso_a3"].map(iso_to_rice50)
mask_missing = world["rice50"].isna()
world.loc[mask_missing, "rice50"] = world.loc[mask_missing, "name"].map(_name_fallback)

for policy, region_ecr in ecr_end.items():
    world[f"ecr_{policy}"] = world["rice50"].map(region_ecr)

n_mapped = world["rice50"].notna().sum()
print(f"Countries mapped to RICE50 regions: {n_mapped}/{len(world)}")

# ── Dissolve into 57 RICE50 region polygons ───────────────────────────────────
ecr_cols_map = [f"ecr_{p}" for p in ecr_end.keys()]

regions_gdf = (
    world[world["rice50"].notna()]
    .dissolve(by="rice50", aggfunc="first")
    .reset_index()
)[["rice50", "geometry"] + ecr_cols_map]

import geopandas as _gpd
if not isinstance(regions_gdf, _gpd.GeoDataFrame):
    regions_gdf = _gpd.GeoDataFrame(regions_gdf, geometry="geometry", crs=world.crs)

print(f"Dissolved to {len(regions_gdf)} RICE50 regions")


In [ ]:
# ── World map — end-of-horizon ECR snapshot (dynamic grid, one panel per policy) ──

panel_specs = [
    (f"ecr_{label}", f"{label}\nECR snapshot ({snap_year})")
    for label in ecr_end.keys()
]

# Check that all required ECR columns exist
missing_cols = [col for col, _ in panel_specs if col not in regions_gdf.columns]
if missing_cols:
    raise KeyError(f"Missing ECR columns in regions_gdf: {missing_cols}")

all_vals = pd.concat([regions_gdf[col].dropna() for col, _ in panel_specs])
vmin_all, vmax_all = all_vals.min(), all_vals.max()

_cmap = plt.cm.YlOrRd
_norm = mcolors.Normalize(vmin=vmin_all, vmax=vmax_all)

# Dynamic grid: works for 5 policies, leaves one empty panel
n_panels = len(panel_specs)
n_cols = 2
n_rows = int(np.ceil(n_panels / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes_flat = axes.flatten()

for ax, (col, title) in zip(axes_flat, panel_specs):
    world.plot(ax=ax, color="0.88")
    world.boundary.plot(ax=ax, color="0.7", linewidth=0.2)

    colors = [
        _cmap(_norm(v)) if pd.notna(v) else "0.88"
        for v in regions_gdf[col]
    ]

    regions_gdf.plot(ax=ax, color=colors)
    regions_gdf.boundary.plot(ax=ax, color="0.4", linewidth=0.5)

    sm = plt.cm.ScalarMappable(cmap=_cmap, norm=_norm)
    sm.set_array([])

    cbar = fig.colorbar(
        sm,
        ax=ax,
        orientation="horizontal",
        pad=0.02,
        shrink=0.75,
        aspect=35,
    )
    cbar.set_label("Emission Control Rate (0–1)", fontsize=8)

    ax.set_title(title, fontsize=11, fontweight="bold", pad=5)
    ax.set_axis_off()

# Hide unused axes, e.g. 6th panel if there are 5 policies
for ax in axes_flat[len(panel_specs):]:
    ax.set_axis_off()

fig.suptitle(
    f"Regional ECR at End of Simulation Horizon ({snap_year}) — JUSTICE RICE50 Model\n"
    "57 aggregate regions; shared colour scale across selected policies",
    fontsize=12,
    y=1.01,
)

plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, "ecr_regional_map_5_policies.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("Figure saved:", plot_path)

run reeval 


## 7. Robustness analysis and policy re-evaluation

Use this section after the selected policies have been re-evaluated across the uncertainty ensemble. The aim is to test whether the policy still satisfies South Africa’s climate and transition-burden criteria.


In [ ]:
# ============================================================
# ASSIGNMENT 8 — ROBUSTNESS ANALYSIS FOR SOUTH AFRICA
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    import matplotlib
    matplotlib.use("Agg")

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

# ------------------------------------------------------------
# 1. Load re-evaluation results
# ------------------------------------------------------------

OBJECTIVES_REEVAL = [
    "welfare",
    "fraction_above_threshold",
    "global_temperature_2100",
    "max_global_temperature",
    "welfare_loss_damage",
    "welfare_loss_abatement",
    "abatement_burden",
    "zaf_mean_abatement_burden",
    "zaf_mean_damage_fraction",
    "zaf_mean_net_output_ratio",
]

obj_idx = {name: i for i, name in enumerate(OBJECTIVES_REEVAL)}

# Use your project results directory if already defined
try:
    RESULTS_ROOT = Path(RESULTS_DIR)
except NameError:
    RESULTS_ROOT = Path("results")

# Load latest prioritarian zafmetrics file
reeval_files = sorted(RESULTS_ROOT.glob("reeval_prioritarian_zafmetrics_*p_*s.npy"))

if not reeval_files:
    raise FileNotFoundError(
        f"No re-evaluation files found in {RESULTS_ROOT}. "
        "Expected pattern: reeval_prioritarian_zafmetrics_*p_*s.npy"
    )

RESULTS_PATH = reeval_files[-1]
results = np.load(RESULTS_PATH)

N_POLICIES, N_SCENARIOS, N_OBJECTIVES = results.shape

print(f"Loaded: {RESULTS_PATH}")
print(f"Shape: {results.shape} = policies × scenarios × objectives")

if N_OBJECTIVES != len(OBJECTIVES_REEVAL):
    raise ValueError(
        f"Expected {len(OBJECTIVES_REEVAL)} objectives, but results has {N_OBJECTIVES}."
    )

In [ ]:
# ============================================================
# 2. South Africa satisficing thresholds
# ============================================================

SATISFICING_OBJECTIVES = [
    "global_temperature_2100",
    "zaf_mean_abatement_burden",
    "zaf_mean_damage_fraction",
    "zaf_mean_net_output_ratio",
]

THRESHOLDS = {
    "global_temperature_2100": 2.0,
    "zaf_mean_abatement_burden": 0.05,
    "zaf_mean_damage_fraction": 0.05,
    "zaf_mean_net_output_ratio": 0.95,
}

DIRECTIONS = {
    "global_temperature_2100": "<=",
    "zaf_mean_abatement_burden": "<=",
    "zaf_mean_damage_fraction": "<=",
    "zaf_mean_net_output_ratio": ">=",
}

threshold_table = pd.DataFrame({
    "objective": SATISFICING_OBJECTIVES,
    "direction": [DIRECTIONS[o] for o in SATISFICING_OBJECTIVES],
    "threshold": [THRESHOLDS[o] for o in SATISFICING_OBJECTIVES],
    "interpretation": [
        "Global temperature in 2100 should not exceed 2°C.",
        "South Africa's mean abatement burden should not exceed 5% of gross economic output.",
        "South Africa's mean damage fraction should not exceed 5%.",
        "South Africa's net economic output should remain at least 95% of gross output.",
    ],
})

display(threshold_table)

In [ ]:
# ============================================================
# 3. Compute satisficing scores
# ============================================================

def compute_satisficing(results, objectives, thresholds, directions, obj_idx):
    n_policies, n_scenarios, _ = results.shape

    per_objective_meets = np.zeros(
        (n_policies, n_scenarios, len(objectives)),
        dtype=bool,
    )

    for j, obj in enumerate(objectives):
        values = results[:, :, obj_idx[obj]]

        if directions[obj] == "<=":
            per_objective_meets[:, :, j] = values <= thresholds[obj]
        elif directions[obj] == ">=":
            per_objective_meets[:, :, j] = values >= thresholds[obj]
        else:
            raise ValueError(f"Unknown direction for {obj}: {directions[obj]}")

    joint_meets = per_objective_meets.all(axis=2)

    sat_score = joint_meets.mean(axis=1)
    per_obj_policy_scores = per_objective_meets.mean(axis=1)

    summary = pd.DataFrame({
        "policy": [f"P{i}" for i in range(n_policies)],
        "satisficing_score": sat_score,
    }).sort_values("satisficing_score", ascending=False).reset_index(drop=True)

    objective_summary = pd.DataFrame({
        "objective": objectives,
        "overall_satisficing_rate": per_objective_meets.mean(axis=(0, 1)),
    }).sort_values("overall_satisficing_rate")

    return summary, objective_summary, per_obj_policy_scores, joint_meets


sat_summary, objective_summary, per_obj_scores, joint_meets = compute_satisficing(
    results,
    SATISFICING_OBJECTIVES,
    THRESHOLDS,
    DIRECTIONS,
    obj_idx,
)

print("Top 10 policies by South Africa satisficing score:")
display(sat_summary.head(10))

print("Most difficult objectives to satisfy:")
display(objective_summary)

print(f"Mean satisficing score: {sat_summary['satisficing_score'].mean():.3f}")
print(f"Best satisficing score: {sat_summary['satisficing_score'].max():.3f}")


In [ ]:
# ============================================================
# 5. Minimax regret
# ============================================================

def compute_minimax_regret(results, objectives, directions, obj_idx):
    selected = np.stack(
        [results[:, :, obj_idx[obj]] for obj in objectives],
        axis=2,
    )

    n_policies, n_scenarios, n_objectives = selected.shape
    regret = np.zeros_like(selected, dtype=float)

    for j, obj in enumerate(objectives):
        values = selected[:, :, j]

        if directions[obj] == "<=":
            ideal = np.nanmin(values, axis=0)
            anti_ideal = np.nanmax(values, axis=0)
            denom = anti_ideal - ideal
            regret[:, :, j] = (values - ideal) / np.maximum(denom, 1e-12)

        elif directions[obj] == ">=":
            ideal = np.nanmax(values, axis=0)
            anti_ideal = np.nanmin(values, axis=0)
            denom = ideal - anti_ideal
            regret[:, :, j] = (ideal - values) / np.maximum(denom, 1e-12)

        else:
            raise ValueError(f"Unknown direction for {obj}: {directions[obj]}")

    regret = np.nan_to_num(regret, nan=1.0, posinf=1.0, neginf=1.0)

    total_regret = regret.sum(axis=2)
    max_regret = total_regret.max(axis=1)
    mean_regret = total_regret.mean(axis=1)

    regret_summary = pd.DataFrame({
        "policy": [f"P{i}" for i in range(n_policies)],
        "max_regret": max_regret,
        "mean_regret": mean_regret,
    }).sort_values("max_regret").reset_index(drop=True)

    return regret_summary, regret, total_regret


regret_summary, regret, total_regret = compute_minimax_regret(
    results,
    SATISFICING_OBJECTIVES,
    DIRECTIONS,
    obj_idx,
)

print("Top 10 policies by minimax regret:")
display(regret_summary.head(10))

best_minimax_policy = regret_summary.iloc[0]["policy"]
print("Best minimax-regret policy:", best_minimax_policy)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# 6. Bereken SA Satisficing Robustness direct uit 'results'
# ============================================================

# Haal de juiste indices op voor de 4 Zuid-Afrika criteria
idx_temp   = obj_idx["global_temperature_2100"]
idx_abate  = obj_idx["zaf_mean_abatement_burden"]
idx_damage = obj_idx["zaf_mean_damage_fraction"]
idx_output = obj_idx["zaf_mean_net_output_ratio"]

# Bereken per policy en scenario of ze aan de drempelwaarden voldoen
pass_temp   = results[:, :, idx_temp] <= 2.0
pass_abate  = results[:, :, idx_abate] <= 0.05
pass_damage = results[:, :, idx_damage] <= 0.05
pass_output = results[:, :, idx_output] >= 0.95

# Een scenario is 'satisficing' als ALLE 4 voorwaarden kloppen
pass_all = pass_temp & pass_abate & pass_damage & pass_output

# De robuustheid is de fractie (gemiddelde) van de scenario's die geslaagd zijn
sa_robustness_scores = pass_all.mean(axis=1)

# Maak een dataframe van deze scores zodat we ze veilig kunnen koppelen
# aan jouw al gesorteerde regret_summary
robustness_df = pd.DataFrame({
    "policy": [f"P{i}" for i in range(results.shape[0])],
    "sa_robustness_score": sa_robustness_scores
})

# Koppel de zojuist berekende robuustheid aan jouw minimax resultaten
tradeoff_df = pd.merge(regret_summary, robustness_df, on="policy")


# ============================================================
# 7. Trade-off plot: Minimax Regret vs. Satisficing Robustness
# ============================================================

plt.figure(figsize=(10, 7))

# Scatterplot maken zonder kleurenschaal (alle bolletjes 1 kleur)
scatter = plt.scatter(
    tradeoff_df["max_regret"],
    tradeoff_df["sa_robustness_score"],
    s=80,
    color='steelblue',   # Hier bepalen we de vaste kleur
    alpha=0.8,
    edgecolor='black',
    linewidth=0.5
)

plt.xlabel("Minimax Regret (Lower is better)")
plt.ylabel("Joint Satisficing Robustness (Higher is better)")
plt.title("South Africa: Satisficing Robustness vs Minimax Regret")
plt.grid(alpha=0.3)

# Markeer het perfecte (meestal onhaalbare) punt in de grafiek
plt.plot(0, 1, marker='.', color='red', markersize=15, markeredgecolor='black', label='Ideal Point (No Regret, 100% Robust)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6. Combined robustness ranking
# ============================================================

from pathlib import Path

combined = sat_summary.merge(
    regret_summary,
    on="policy",
    how="left",
)

combined["satisficing_rank"] = combined["satisficing_score"].rank(
    ascending=False,
    method="min",
)

combined["regret_rank"] = combined["max_regret"].rank(
    ascending=True,
    method="min",
)

combined["combined_rank"] = (
    combined["satisficing_rank"]
    + combined["regret_rank"]
)

combined = combined.sort_values(
    ["combined_rank", "satisficing_score", "max_regret"],
    ascending=[True, False, True],
).reset_index(drop=True)

print("Combined robustness ranking:")
display(combined.head(15))

best_overall_policy = combined.iloc[0]["policy"]
print("Best combined robustness policy:", best_overall_policy)


# ============================================================
# SAVE COMBINED ROBUSTNESS RANKING
# ============================================================

# Use RESULTS_DIR if it already exists; otherwise use ./results
output_dir = (
    Path(RESULTS_DIR)
    if "RESULTS_DIR" in globals()
    else Path("results")
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

combined_output_path = (
    output_dir / "combined_robustness_ranking.csv"
)

combined.to_csv(
    combined_output_path,
    index=False,
)

print(
    "Saved combined robustness ranking to:",
    combined_output_path.resolve(),
)

In [ ]:
import copy
# ── VISUALISATION ─────────────────────────────────────────────────────────────

# FIX 0: Selecteer alleen de 4 satisficing criteria
# PAS DEZE INDICES AAN als jullie 4 criteria niet de eerste 4 kolommen zijn.
SATISFICING_IDX = [0, 1, 2, 3]

results_crit = results[:, :, SATISFICING_IDX]   # shape: (n_policies, n_scenarios, 4)

# Labels voor alleen deze 4 criteria
# Als OBJ_LABELS al maar 4 labels heeft, laat dit zo:
OBJ_LABELS_CRIT = [
    "Temperature 2100",
    "SA abatement burden",
    "SA damage fraction",
    "SA net output ratio",
]


# Als OBJ_LABELS nog 10 labels heeft, gebruik dan deze regel in plaats van hierboven:
# OBJ_LABELS_CRIT = [OBJ_LABELS[i] for i in SATISFICING_IDX]


# FIX 1: Haal het aantal policies dynamisch uit de gefilterde resultaten
N_POLICIES = results_crit.shape[0]

# FIX 3: Thresholds alleen voor de 4 criteria
THRESHOLD_PERCENTILES = [95, 90, 5, 5]

assert results_crit.shape[-1] == len(THRESHOLD_PERCENTILES), (
    f"Mismatch: results_crit has {results_crit.shape[-1]} criteria, "
    f"but THRESHOLD_PERCENTILES has {len(THRESHOLD_PERCENTILES)} values."
)

assert len(OBJ_LABELS_CRIT) == results_crit.shape[-1], (
    f"Mismatch: OBJ_LABELS_CRIT has {len(OBJ_LABELS_CRIT)} labels, "
    f"but results_crit has {results_crit.shape[-1]} criteria."
)

flat = results_crit.reshape(-1, results_crit.shape[-1])

thresholds_arr = np.array([
    np.nanpercentile(flat[:, i], THRESHOLD_PERCENTILES[i])
    for i in range(results_crit.shape[-1])
])


def _draw_heatmap(ax, data, cmap, vmin, vmax, row_labels, col_labels, title, cbar_label):
    """imshow-based heatmap (avoids seaborn/matplotlib 3.14 deepcopy incompatibility)."""
    im = ax.imshow(
        data,
        aspect="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest"
    )

    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, fontsize=9, rotation=20, ha="right")

    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=7)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            tc = "white" if v > (vmax - vmin) * 0.6 + vmin else "black"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color=tc)

    plt.colorbar(im, ax=ax, label=cbar_label, shrink=0.8)
    ax.set_title(title, fontsize=10)


# ── Figure 1: Satisficing heatmap ─────────────────────────────────────────────

# Per-objective satisficing: alleen over de 4 criteria
per_obj_sat = np.nanmean(
    results_crit <= thresholds_arr[np.newaxis, np.newaxis, :],
    axis=1
)  # shape: (n_policies, 4)

# Joint satisficing score: policy voldoet aan ALLE 4 criteria tegelijk
joint_sat = np.all(
    results_crit <= thresholds_arr[np.newaxis, np.newaxis, :],
    axis=2
)  # shape: (n_policies, n_scenarios)

sat_score = np.nanmean(joint_sat, axis=1)  # shape: (n_policies,)

# Sort policies by joint satisficing score
sat_order = np.argsort(-sat_score)

fig, ax = plt.subplots(figsize=(6, 15))

_draw_heatmap(
    ax,
    per_obj_sat[sat_order],
    "YlGn",
    0,
    1,
    [f"P{sat_order[i]}" for i in range(N_POLICIES)],
    OBJ_LABELS_CRIT,
    "Per-objective satisficing rate\n(sorted by joint satisficing score)",
    f"Fraction of {results_crit.shape[1]} scenarios meeting threshold",
)

ax.set_xlabel("Criterion")
ax.set_ylabel("Policy (sorted)")
plt.tight_layout()

display(fig)

plt.savefig(
    os.path.join(_PLOTS_DIR, "a07_satisficing_heatmap.png"),
    dpi=150,
    bbox_inches="tight"
)

plt.close(fig)

print("Figure saved: plots/a07_satisficing_heatmap.png")

In [ ]:
import copy

# ── VISUALISATION ─────────────────────────────────────────────────────────────

# FIX 0: Selecteer alleen de 4 satisficing criteria
SATISFICING_IDX = [0, 1, 2, 3]

results_crit = results[:, :, SATISFICING_IDX]
# shape: (n_policies, n_scenarios, 4)

OBJ_LABELS_CRIT = [
    "Temperature 2100",
    "SA abatement burden",
    "SA damage fraction",
    "SA net output ratio",
]

# Number of policies in the full result set
N_POLICIES = results_crit.shape[0]

# Number of policies to show
TOP_N = min(15, N_POLICIES)

# Thresholds for the 4 criteria
THRESHOLD_PERCENTILES = [95, 90, 5, 5]

assert results_crit.shape[-1] == len(THRESHOLD_PERCENTILES), (
    f"Mismatch: results_crit has {results_crit.shape[-1]} criteria, "
    f"but THRESHOLD_PERCENTILES has {len(THRESHOLD_PERCENTILES)} values."
)

assert len(OBJ_LABELS_CRIT) == results_crit.shape[-1], (
    f"Mismatch: OBJ_LABELS_CRIT has {len(OBJ_LABELS_CRIT)} labels, "
    f"but results_crit has {results_crit.shape[-1]} criteria."
)

flat = results_crit.reshape(-1, results_crit.shape[-1])

thresholds_arr = np.array([
    np.nanpercentile(flat[:, i], THRESHOLD_PERCENTILES[i])
    for i in range(results_crit.shape[-1])
])


def _draw_heatmap(
    ax,
    data,
    cmap,
    vmin,
    vmax,
    row_labels,
    col_labels,
    title,
    cbar_label,
):
    """imshow-based heatmap."""

    im = ax.imshow(
        data,
        aspect="auto",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
    )

    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(
        col_labels,
        fontsize=9,
        rotation=20,
        ha="right",
    )

    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=8)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]

            tc = (
                "white"
                if v > (vmax - vmin) * 0.6 + vmin
                else "black"
            )

            ax.text(
                j,
                i,
                f"{v:.2f}",
                ha="center",
                va="center",
                fontsize=7,
                color=tc,
            )

    plt.colorbar(
        im,
        ax=ax,
        label=cbar_label,
        shrink=0.8,
    )

    ax.set_title(title, fontsize=10)


# ── Calculate satisficing scores ──────────────────────────────────────────────

per_obj_sat = np.nanmean(
    results_crit <= thresholds_arr[np.newaxis, np.newaxis, :],
    axis=1,
)
# shape: (n_policies, 4)

joint_sat = np.all(
    results_crit <= thresholds_arr[np.newaxis, np.newaxis, :],
    axis=2,
)
# shape: (n_policies, n_scenarios)

sat_score = np.nanmean(
    joint_sat,
    axis=1,
)
# shape: (n_policies,)

# Sort all policies by joint satisficing score
sat_order = np.argsort(-sat_score)

# Keep only the top 15
top_policy_idx = sat_order[:TOP_N]

print(f"Showing top {TOP_N} policies:")
for rank, policy_idx in enumerate(top_policy_idx, start=1):
    print(
        f"{rank:2d}. P{policy_idx} "
        f"| joint satisficing score = {sat_score[policy_idx]:.3f}"
    )


# ── Figure: top-15 satisficing heatmap ────────────────────────────────────────

fig_height = max(5, 0.42 * TOP_N)

fig, ax = plt.subplots(
    figsize=(7, fig_height)
)

_draw_heatmap(
    ax=ax,
    data=per_obj_sat[top_policy_idx],
    cmap="YlGn",
    vmin=0,
    vmax=1,
    row_labels=[
        f"P{policy_idx}"
        for policy_idx in top_policy_idx
    ],
    col_labels=OBJ_LABELS_CRIT,
    title=(
        f"Per-objective satisficing rate\n"
        f"Top {TOP_N} policies by joint satisficing score"
    ),
    cbar_label=(
        f"Fraction of {results_crit.shape[1]} "
        f"scenarios meeting threshold"
    ),
)

ax.set_xlabel("Criterion")
ax.set_ylabel("Policy, ranked by joint satisficing score")

plt.tight_layout()

plot_path = os.path.join(
    _PLOTS_DIR,
    "a07_satisficing_heatmap_top15.png",
)

plt.savefig(
    plot_path,
    dpi=150,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(f"Figure saved: {plot_path}")

In [ ]:
# ── SATISFICING HEATMAP VOOR 4 THRESHOLD-CRITERIA ────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Alle 10 model outcomes, in exact dezelfde volgorde als in results[:, :, :]
OBJECTIVES = [
    "welfare",
    "fraction_above_threshold",
    "global_temperature_2100",
    "max_global_temperature",
    "welfare_loss_damage",
    "welfare_loss_abatement",
    "abatement_burden",
    "zaf_mean_abatement_burden",
    "zaf_mean_damage_fraction",
    "zaf_mean_net_output_ratio",
]

# Alleen deze 4 gebruiken we als satisficing criteria
THRESHOLDS = {
    "global_temperature_2100": 2.0,
    "zaf_mean_abatement_burden": 0.05,
    "zaf_mean_damage_fraction": 0.05,
    "zaf_mean_net_output_ratio": 0.95,
}

# Alleen de objectives waarvoor jullie een threshold hebben
SATISFICING_OBJECTIVES = list(THRESHOLDS.keys())

# Vind de juiste kolommen in results
SATISFICING_IDX = [OBJECTIVES.index(obj) for obj in SATISFICING_OBJECTIVES]

# Selecteer alleen deze 4 criteria
results_crit = results[:, :, SATISFICING_IDX]

# Thresholds in dezelfde volgorde als SATISFICING_OBJECTIVES
thresholds_arr = np.array([
    THRESHOLDS[obj] for obj in SATISFICING_OBJECTIVES
])

# Richting per criterium:
# True  = lager is beter, dus <= threshold
# False = hoger is beter, dus >= threshold
LESS_IS_BETTER = np.array([
    True,   # global_temperature_2100 <= 2.0
    True,   # zaf_mean_abatement_burden <= 0.05
    True,   # zaf_mean_damage_fraction <= 0.05
    False,  # zaf_mean_net_output_ratio >= 0.95
])

# Check of alles dezelfde lengte heeft
assert results_crit.shape[-1] == len(SATISFICING_OBJECTIVES)
assert results_crit.shape[-1] == len(thresholds_arr)
assert results_crit.shape[-1] == len(LESS_IS_BETTER)

# Maak satisficing matrix
satisficing = np.empty_like(results_crit, dtype=bool)

for i in range(results_crit.shape[-1]):
    if LESS_IS_BETTER[i]:
        satisficing[:, :, i] = results_crit[:, :, i] <= thresholds_arr[i]
    else:
        satisficing[:, :, i] = results_crit[:, :, i] >= thresholds_arr[i]

# Per policy: percentage scenario's waarin elk criterium gehaald wordt
policy_objective_rates = np.nanmean(satisficing, axis=1)

# Joint satisficing: policy haalt alle 4 criteria tegelijk
joint_sat = np.all(satisficing, axis=2)
sat_score = np.nanmean(joint_sat, axis=1)

# DataFrame voor heatmap
heatmap_df = pd.DataFrame(
    policy_objective_rates,
    columns=SATISFICING_OBJECTIVES,
    index=[f"P{i}" for i in range(results.shape[0])]
)

heatmap_df["joint_sat"] = sat_score
heatmap_df = heatmap_df.sort_values("joint_sat", ascending=False)

# Plot
plt.figure(figsize=(10, max(6, 0.3 * results.shape[0])))

sns.heatmap(
    heatmap_df[SATISFICING_OBJECTIVES],
    vmin=0,
    vmax=1,
    cmap="viridis",
    annot=True,
    fmt=".2f",
    linewidths=0.5
)

plt.xlabel("Satisficing criterion")
plt.ylabel("Policy")
plt.title("Per-objective satisficing rate by policy")
plt.tight_layout()
plt.show()

In [ ]:
#PRIM

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from ema_workbench.analysis import prim

In [ ]:
# ============================================================
# LOAD RESULTS AND EXPERIMENTS
# ============================================================
results = np.load(
    os.path.join(
        RESULTS_DIR,
        "reeval_prioritarian_zafmetrics_134p_1000s.npy"
    )
)

experiments = pd.read_csv(
    os.path.join(
        RESULTS_DIR,
        "reeval_prioritarian_zafmetrics_134p_1000s_experiments.csv"
    )
)

print("results shape:", results.shape)
print("experiments shape:", experiments.shape)

n_policies, n_scenarios, n_objectives = results.shape

print("n_policies:", n_policies)
print("n_scenarios:", n_scenarios)
print("n_objectives:", n_objectives)

display(experiments.head())

In [ ]:
OBJECTIVES = [
    "welfare",
    "fraction_above_threshold",
    "global_temperature_2100",
    "max_global_temperature",
    "welfare_loss_damage",
    "welfare_loss_abatement",
    "abatement_burden",
    "zaf_mean_abatement_burden",
    "zaf_mean_damage_fraction",
    "zaf_net_output_ratio",
]

assert len(OBJECTIVES) == results.shape[2]

In [ ]:
summary = pd.DataFrame({
    "objective": OBJECTIVES,
    "min": np.nanmin(results, axis=(0, 1)),
    "mean": np.nanmean(results, axis=(0, 1)),
    "max": np.nanmax(results, axis=(0, 1)),
})

display(summary)

In [ ]:
# ============================================================
# DEFINE SOUTH AFRICA SATISFICING CRITERIA
# ============================================================

TEMP_2100_THRESHOLD = 2.0
ZAF_ABATEMENT_BURDEN_THRESHOLD = 0.05
ZAF_DAMAGE_FRACTION_THRESHOLD = 0.05
ZAF_NET_OUTPUT_RATIO_THRESHOLD = 0.95

global_temperature_2100 = results[:, :, OBJECTIVES.index("global_temperature_2100")]
zaf_abatement_burden = results[:, :, OBJECTIVES.index("zaf_mean_abatement_burden")]
zaf_damage_fraction = results[:, :, OBJECTIVES.index("zaf_mean_damage_fraction")]
zaf_net_output_ratio = results[:, :, OBJECTIVES.index("zaf_net_output_ratio")]

SA_satisficing = np.stack(
    [
        global_temperature_2100 <= TEMP_2100_THRESHOLD,
        zaf_abatement_burden <= ZAF_ABATEMENT_BURDEN_THRESHOLD,
        zaf_damage_fraction <= ZAF_DAMAGE_FRACTION_THRESHOLD,
        zaf_net_output_ratio >= ZAF_NET_OUTPUT_RATIO_THRESHOLD,
    ],
    axis=2
)

print("SA_satisficing shape:", SA_satisficing.shape)

In [ ]:
# ============================================================
# ROBUSTNESS PER POLICY
# ============================================================

joint_satisficing = np.all(SA_satisficing, axis=2)
sat_score = joint_satisficing.mean(axis=1)

robustness_df = pd.DataFrame({
    "policy_id": np.arange(n_policies),
    "policy": [f"P{i}" for i in range(n_policies)],
    "joint_satisficing_robustness": sat_score,
})

robustness_df = robustness_df.sort_values(
    "joint_satisficing_robustness",
    ascending=False
)

display(robustness_df.head(15))

In [ ]:
# ============================================================
# BUILD POLICY_METRICS TABLE
# Run this BEFORE the final selected_policy_table cell
# ============================================================

import numpy as np
import pandas as pd

# results should have shape: (n_policies, n_scenarios, n_outcomes)
assert "results" in globals(), "results is not defined. Run the re-evaluation loading cell first."
assert results.ndim == 3, f"Expected results to be 3D, got shape {results.shape}"

N_POLICIES = results.shape[0]

# ---- Outcome indices ----
# Pas deze indices aan als jouw outcome-volgorde anders is.
TEMP_IDX = 0
ABATEMENT_IDX = 1
DAMAGE_IDX = 2
NET_OUTPUT_IDX = 3
WELFARE_IDX = 4 if results.shape[2] > 4 else None

temp = results[:, :, TEMP_IDX]
abatement = results[:, :, ABATEMENT_IDX]
damage = results[:, :, DAMAGE_IDX]
net_output = results[:, :, NET_OUTPUT_IDX]
welfare = results[:, :, WELFARE_IDX] if WELFARE_IDX is not None else np.full((N_POLICIES, results.shape[1]), np.nan)

# ---- Thresholds for South Africa satisficing ----
# Gebruik hier dezelfde waarden als in jullie analyse/plots.
TEMP_THRESHOLD = 2.0
ZAF_ABATEMENT_BURDEN_THRESHOLD = 0.05
ZAF_DAMAGE_FRACTION_THRESHOLD = 0.05
ZAF_NET_OUTPUT_RATIO_THRESHOLD = 0.95

satisficing = (
    (temp <= TEMP_THRESHOLD)
    & (abatement <= ZAF_ABATEMENT_BURDEN_THRESHOLD)
    & (damage <= ZAF_DAMAGE_FRACTION_THRESHOLD)
    & (net_output >= ZAF_NET_OUTPUT_RATIO_THRESHOLD)
)

satisficing_score = satisficing.mean(axis=1)

# ---- Regret calculation over the 4 South Africa criteria ----
def normalized_regret_minimize(arr):
    best = np.nanmin(arr, axis=0, keepdims=True)
    worst = np.nanmax(arr, axis=0, keepdims=True)
    scale = np.where((worst - best) == 0, 1, worst - best)
    return (arr - best) / scale

def normalized_regret_maximize(arr):
    best = np.nanmax(arr, axis=0, keepdims=True)
    worst = np.nanmin(arr, axis=0, keepdims=True)
    scale = np.where((best - worst) == 0, 1, best - worst)
    return (best - arr) / scale

regret_temp = normalized_regret_minimize(temp)
regret_abatement = normalized_regret_minimize(abatement)
regret_damage = normalized_regret_minimize(damage)
regret_net_output = normalized_regret_maximize(net_output)

combined_regret = np.nanmean(
    np.stack(
        [regret_temp, regret_abatement, regret_damage, regret_net_output],
        axis=2
    ),
    axis=2
)

# ---- Policy ids ----
# Belangrijk: als je echte policy_ids al bestaan, gebruik die.
if "policy_ids" in globals():
    policy_id_values = list(policy_ids)
else:
    policy_id_values = list(range(N_POLICIES))

policy_metrics = pd.DataFrame({
    "policy_id": policy_id_values,
    "policy": [f"P{pid}" for pid in policy_id_values],

    "satisficing_score": satisficing_score,
    "max_regret": np.nanmax(combined_regret, axis=1),
    "mean_regret": np.nanmean(combined_regret, axis=1),

    "median_global_temperature_2100": np.nanmedian(temp, axis=1),
    "p90_global_temperature_2100": np.nanpercentile(temp, 90, axis=1),

    "median_zaf_mean_abatement_burden": np.nanmedian(abatement, axis=1),
    "p95_zaf_mean_abatement_burden": np.nanpercentile(abatement, 95, axis=1),

    "median_zaf_mean_damage_fraction": np.nanmedian(damage, axis=1),
    "p95_zaf_mean_damage_fraction": np.nanpercentile(damage, 95, axis=1),

    "median_zaf_mean_net_output_ratio": np.nanmedian(net_output, axis=1),
    "p05_zaf_mean_net_output_ratio": np.nanpercentile(net_output, 5, axis=1),

    "mean_welfare": np.nanmean(welfare, axis=1),
})

# Optional useful ranking columns
policy_metrics["rank_satisficing"] = policy_metrics["satisficing_score"].rank(
    ascending=False, method="min"
)
policy_metrics["rank_max_regret"] = policy_metrics["max_regret"].rank(
    ascending=True, method="min"
)
policy_metrics["combined_rank"] = (
    policy_metrics["rank_satisficing"] + policy_metrics["rank_max_regret"]
)

display(policy_metrics.head())
print("policy_metrics created with shape:", policy_metrics.shape)

In [ ]:
###WELKE POLICIES GAAN WE DOEN?###

# ============================================================
# AUTOMATIC POLICY ARCHETYPE SELECTION FROM THE .NPY FILE
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

assert results.ndim == 3, (
    f"Expected results with shape "
    f"(policies, scenarios, outcomes), got {results.shape}"
)

n_policies, n_scenarios, n_outcomes = results.shape

print("Loaded results:", results.shape)


# ============================================================
# 1. DEFINE OUTCOME ORDER
# ============================================================

OBJECTIVES = [
    "welfare",
    "fraction_above_threshold",
    "global_temperature_2100",
    "max_global_temperature",
    "welfare_loss_damage",
    "welfare_loss_abatement",
    "abatement_burden",
    "zaf_mean_abatement_burden",
    "zaf_mean_damage_fraction",
    "zaf_mean_net_output_ratio",
]

assert len(OBJECTIVES) == n_outcomes, (
    f"The .npy file contains {n_outcomes} outcomes, "
    f"but OBJECTIVES contains {len(OBJECTIVES)} names."
)

outcome_idx = {
    name: i for i, name in enumerate(OBJECTIVES)
}


# ============================================================
# 2. EXTRACT SOUTH AFRICA OUTCOMES
# ============================================================

temperature_2100 = results[
    :, :, outcome_idx["global_temperature_2100"]
]

zaf_abatement = results[
    :, :, outcome_idx["zaf_mean_abatement_burden"]
]

zaf_damage = results[
    :, :, outcome_idx["zaf_mean_damage_fraction"]
]

zaf_net_output = results[
    :, :, outcome_idx["zaf_mean_net_output_ratio"]
]


# ============================================================
# 3. CALCULATE SATISFICING ROBUSTNESS
# ============================================================

TEMP_THRESHOLD = 2.0
ABATEMENT_THRESHOLD = 0.05
DAMAGE_THRESHOLD = 0.05
NET_OUTPUT_THRESHOLD = 0.95

satisfies_temperature = (
    temperature_2100 <= TEMP_THRESHOLD
)

satisfies_abatement = (
    zaf_abatement <= ABATEMENT_THRESHOLD
)

satisfies_damage = (
    zaf_damage <= DAMAGE_THRESHOLD
)

satisfies_net_output = (
    zaf_net_output >= NET_OUTPUT_THRESHOLD
)

joint_satisficing = (
    satisfies_temperature
    & satisfies_abatement
    & satisfies_damage
    & satisfies_net_output
)

# Share of scenarios in which all four criteria are satisfied
satisficing_score = joint_satisficing.mean(axis=1)


# ============================================================
# 4. CALCULATE NORMALISED REGRET
# ============================================================

def normalised_regret_lower_better(values):
    """
    Scenario-specific normalised regret for an outcome
    where lower values are preferred.
    """
    scenario_best = np.nanmin(values, axis=0)
    scenario_worst = np.nanmax(values, axis=0)

    denominator = scenario_worst - scenario_best
    denominator = np.where(
        denominator > 0,
        denominator,
        1.0,
    )

    return (
        values - scenario_best
    ) / denominator


def normalised_regret_higher_better(values):
    """
    Scenario-specific normalised regret for an outcome
    where higher values are preferred.
    """
    scenario_best = np.nanmax(values, axis=0)
    scenario_worst = np.nanmin(values, axis=0)

    denominator = scenario_best - scenario_worst
    denominator = np.where(
        denominator > 0,
        denominator,
        1.0,
    )

    return (
        scenario_best - values
    ) / denominator


regret_temperature = normalised_regret_lower_better(
    temperature_2100
)

regret_abatement = normalised_regret_lower_better(
    zaf_abatement
)

regret_damage = normalised_regret_lower_better(
    zaf_damage
)

regret_net_output = normalised_regret_higher_better(
    zaf_net_output
)

regret_stack = np.stack(
    [
        regret_temperature,
        regret_abatement,
        regret_damage,
        regret_net_output,
    ],
    axis=2,
)

# Average regret across the four criteria in each scenario
scenario_regret = np.nanmean(
    regret_stack,
    axis=2,
)

# Worst-case regret across scenarios
max_regret = np.nanmax(
    scenario_regret,
    axis=1,
)

mean_regret = np.nanmean(
    scenario_regret,
    axis=1,
)


# ============================================================
# 5. BUILD POLICY_METRICS FOR ALL POLICIES
# ============================================================

policy_metrics = pd.DataFrame({
    "policy_id": np.arange(n_policies, dtype=int),
    "policy": [f"P{i}" for i in range(n_policies)],

    "satisficing_score": satisficing_score,
    "max_regret": max_regret,
    "mean_regret": mean_regret,

    "median_temperature_2100": np.nanmedian(
        temperature_2100,
        axis=1,
    ),

    "median_zaf_abatement_burden": np.nanmedian(
        zaf_abatement,
        axis=1,
    ),

    "median_zaf_damage_fraction": np.nanmedian(
        zaf_damage,
        axis=1,
    ),

    "median_zaf_net_output_ratio": np.nanmedian(
        zaf_net_output,
        axis=1,
    ),

    "p90_zaf_abatement_burden": np.nanpercentile(
        zaf_abatement,
        90,
        axis=1,
    ),

    "p90_zaf_damage_fraction": np.nanpercentile(
        zaf_damage,
        90,
        axis=1,
    ),

    "p10_zaf_net_output_ratio": np.nanpercentile(
        zaf_net_output,
        10,
        axis=1,
    ),
})


# ============================================================
# 6. CREATE ROBUSTNESS RANKS
# ============================================================

policy_metrics["satisficing_rank"] = (
    policy_metrics["satisficing_score"]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

policy_metrics["minimax_regret_rank"] = (
    policy_metrics["max_regret"]
    .rank(
        ascending=True,
        method="min",
    )
    .astype(int)
)

# Equal weighting of the two robustness rankings
policy_metrics["combined_rank_sum"] = (
    policy_metrics["satisficing_rank"]
    + policy_metrics["minimax_regret_rank"]
)

policy_metrics["combined_rank"] = (
    policy_metrics["combined_rank_sum"]
    .rank(
        ascending=True,
        method="min",
    )
    .astype(int)
)


# ============================================================
# 7. DEFINE THE ROBUST CANDIDATE POOL
# ============================================================
#
# The candidate pool is the union of:
# - top policies on satisficing robustness;
# - top policies on minimax regret;
# - top policies on the combined rank.
#
# This prevents "robust" from being defined using only one metric.
# ============================================================

TOP_N_PER_METRIC = 20

top_satisficing_ids = set(
    policy_metrics
    .nsmallest(
        TOP_N_PER_METRIC,
        "satisficing_rank",
    )["policy_id"]
)

top_minimax_ids = set(
    policy_metrics
    .nsmallest(
        TOP_N_PER_METRIC,
        "minimax_regret_rank",
    )["policy_id"]
)

top_combined_ids = set(
    policy_metrics
    .nsmallest(
        TOP_N_PER_METRIC,
        "combined_rank",
    )["policy_id"]
)

robust_candidate_ids = (
    top_satisficing_ids
    | top_minimax_ids
    | top_combined_ids
)

robust_policy_metrics = policy_metrics[
    policy_metrics["policy_id"].isin(
        robust_candidate_ids
    )
].copy()

print(
    "Number of policies in robust candidate pool:",
    len(robust_policy_metrics),
)


# ============================================================
# 8. IDENTIFY LOWEST ABATEMENT BURDEN
#    AMONG ROBUST POLICIES
# ============================================================

lowest_abatement_policy = robust_policy_metrics.loc[
    robust_policy_metrics[
        "median_zaf_abatement_burden"
    ].idxmin()
]


# ============================================================
# 9. IDENTIFY HIGHEST NET OUTPUT RATIO
#    AMONG ROBUST POLICIES
# ============================================================

highest_net_output_policy = robust_policy_metrics.loc[
    robust_policy_metrics[
        "median_zaf_net_output_ratio"
    ].idxmax()
]

# ============================================================
# 10. IDENTIFY LOWEST SOUTH AFRICA DAMAGE FRACTION
#     AMONG ROBUST POLICIES
# ============================================================

lowest_damage_policy = robust_policy_metrics.loc[
    robust_policy_metrics[
        "median_zaf_damage_fraction"
    ].idxmin()
]


print("\nLOWEST ABATEMENT BURDEN AMONG ROBUST POLICIES")
print("------------------------------------------------")

print(
    "Policy:",
    lowest_abatement_policy["policy"],
)

print(
    "Median abatement burden:",
    round(
        lowest_abatement_policy[
            "median_zaf_abatement_burden"
        ],
        6,
    ),
)

print(
    "Satisficing score:",
    round(
        lowest_abatement_policy[
            "satisficing_score"
        ],
        4,
    ),
)

print(
    "Maximum regret:",
    round(
        lowest_abatement_policy[
            "max_regret"
        ],
        4,
    ),
)


print("\nHIGHEST NET OUTPUT RATIO AMONG ROBUST POLICIES")
print("------------------------------------------------")

print(
    "Policy:",
    highest_net_output_policy["policy"],
)

print(
    "Median net output ratio:",
    round(
        highest_net_output_policy[
            "median_zaf_net_output_ratio"
        ],
        6,
    ),
)

print(
    "Satisficing score:",
    round(
        highest_net_output_policy[
            "satisficing_score"
        ],
        4,
    ),
)

print(
    "Maximum regret:",
    round(
        highest_net_output_policy[
            "max_regret"
        ],
        4,
    ),
)

print("\nLOWEST DAMAGE FRACTION AMONG ROBUST POLICIES")
print("------------------------------------------------")

print(
    "Policy:",
    lowest_damage_policy["policy"],
)

print(
    "Median South Africa damage fraction:",
    round(
        lowest_damage_policy[
            "median_zaf_damage_fraction"
        ],
        6,
    ),
)

print(
    "Satisficing score:",
    round(
        lowest_damage_policy[
            "satisficing_score"
        ],
        4,
    ),
)

print(
    "Maximum regret:",
    round(
        lowest_damage_policy[
            "max_regret"
        ],
        4,
    ),
)

In [ ]:
# ============================================================
# SELECT LOWEST GLOBAL TEMPERATURE IN 2100
# AMONG THE ROBUST POLICIES
# ============================================================

# Recreate the robust-policy dataframe from the automatic
# robust candidate IDs, ensuring it contains the latest metrics.
robust_policy_metrics = policy_metrics[
    policy_metrics["policy_id"].isin(robust_candidate_ids)
].copy()

# Policy with the lowest median global temperature in 2100
lowest_temperature_policy = robust_policy_metrics.loc[
    robust_policy_metrics["median_temperature_2100"].idxmin()
]

print("LOWEST GLOBAL TEMPERATURE IN 2100 AMONG ROBUST POLICIES")
print("--------------------------------------------------------")
print(f"Policy: {lowest_temperature_policy['policy']}")
print(
    "Median global temperature in 2100: "
    f"{lowest_temperature_policy['median_temperature_2100']:.4f} °C"
)
print(
    "Satisficing score: "
    f"{lowest_temperature_policy['satisficing_score']:.4f}"
)
print(
    "Maximum regret: "
    f"{lowest_temperature_policy['max_regret']:.4f}"
)
print(
    "Combined robustness rank: "
    f"{int(lowest_temperature_policy['combined_rank'])}"
)

Choosing 5 policies for PRIM

In [ ]:
# ============================================================
# FINAL DEFENSIBLE POLICY SELECTION FOR SOUTH AFRICA
# ============================================================


## DEZE AANPASSEN NAAR ##
SELECTED_POLICY_IDS = [93, 22, 44]

selection_labels = {
    93: "Highest South Africa satisficing robustness",
    22: "Best overall combined robustness en highest net output ratio",
    44: "Lowest abatement burden, lower robustness score",
}

In [ ]:
# ============================================================
# PREPARE X FOR PRIM FOR ONE POLICY
# ============================================================

POLICY_ID =  93#int(robustness_df.iloc[0]["policy_id"])  # meest robuuste policy
POLICY_LABEL = f"P{POLICY_ID}"

print("Selected policy:", POLICY_LABEL)

policy_experiments = experiments.loc[
    experiments["policy"] == POLICY_LABEL
].copy()

print("policy_experiments shape:", policy_experiments.shape)

if len(policy_experiments) != n_scenarios:
    raise ValueError(
        f"Expected {n_scenarios} rows for {POLICY_LABEL}, "
        f"but got {len(policy_experiments)}."
    )

display(policy_experiments.head())

In [ ]:
# ============================================================
# SUMMARY TABLE FOR POLICIES 22, 93, AND 44
# ============================================================

from pathlib import Path
import pandas as pd

SELECTED_POLICY_IDS = [22, 93, 44]

# Check that all requested policies exist
available_policy_ids = set(
    policy_metrics["policy_id"].astype(int)
)

missing_policy_ids = [
    policy_id
    for policy_id in SELECTED_POLICY_IDS
    if policy_id not in available_policy_ids
]

assert not missing_policy_ids, (
    f"These policy IDs are missing from policy_metrics: "
    f"{missing_policy_ids}"
)

# Check that all required metrics exist
required_columns = [
    "policy_id",
    "satisficing_score",
    "mean_regret",
    "median_zaf_net_output_ratio",
    "median_temperature_2100",
    "median_zaf_damage_fraction",
    "median_zaf_abatement_burden",
]

missing_columns = [
    column
    for column in required_columns
    if column not in policy_metrics.columns
]

assert not missing_columns, (
    f"These columns are missing from policy_metrics: "
    f"{missing_columns}"
)

# Select policies and preserve the order 22, 93, 44
policy_comparison = (
    policy_metrics
    .set_index("policy_id")
    .loc[SELECTED_POLICY_IDS, required_columns[1:]]
    .reset_index()
)

# Rename columns for a readable table and CSV
policy_comparison = policy_comparison.rename(
    columns={
        "policy_id": "Policy ID",
        "satisficing_score": "Satisficing robustness",
        "mean_regret": "Mean regret",
        "median_zaf_net_output_ratio": "SA net output ratio",
        "median_temperature_2100": "Global temperature 2100",
        "median_zaf_damage_fraction": "SA damage fraction",
        "median_zaf_abatement_burden": "SA abatement burden",
    }
)

# Display a rounded version
display(
    policy_comparison.round(
        {
            "Satisficing robustness": 4,
            "Mean regret": 4,
            "SA net output ratio": 4,
            "Global temperature 2100": 4,
            "SA damage fraction": 4,
            "SA abatement burden": 4,
        }
    )
)

# ============================================================
# SAVE TO CSV
# ============================================================

output_dir = (
    Path(RESULTS_DIR)
    if "RESULTS_DIR" in globals()
    else Path("results")
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

output_path = (
    output_dir / "policy_comparison_P22_P93_P44.csv"
)

policy_comparison.to_csv(
    output_path,
    index=False,
)

print("Saved policy comparison to:")
print(output_path.resolve())

In [ ]:
# ============================================================
# SELECT UNCERTAINTY COLUMNS FOR PRIM
# ============================================================

UNCERTAINTY_COLS = [
    "climate_ensemble_index"
]

X_prim = policy_experiments[UNCERTAINTY_COLS].copy()
X_prim = X_prim.reset_index(drop=True)

print("X_prim shape:", X_prim.shape)
display(X_prim.head())

In [ ]:
# ============================================================
# DEFINE FAILURE LABEL FOR PRIM
# ============================================================
y_failure = ~np.all(SA_satisficing[POLICY_ID, :, :], axis=1)

print(f"Policy {POLICY_LABEL}")
print(f"Failures: {y_failure.sum()} out of {len(y_failure)}")
print(f"Failure rate: {100 * y_failure.mean():.1f}%")

if y_failure.sum() == 0:
    raise ValueError("This policy has no failures. PRIM cannot explain failures.")

if y_failure.sum() == len(y_failure):
    raise ValueError("This policy fails in all scenarios. PRIM cannot distinguish failure conditions.")

In [ ]:
# ============================================================
# RUN PRIM
# ============================================================

from ema_workbench.analysis import prim

prim_alg = prim.Prim(
    X_prim,
    y_failure
)

box = prim_alg.find_box()

In [ ]:
# ============================================================
# PRIM TRADE-OFF PLOT
# ============================================================

box.show_tradeoff()
plt.title(f"PRIM trade-off — satisficing failures for {POLICY_LABEL}")
plt.show()

In [ ]:
# ============================================================
# INSPECT PEELING TRAJECTORY
# ============================================================

traj = box.peeling_trajectory.copy()

display(traj.head())
display(traj.tail())

In [ ]:
# ============================================================
# SELECT PRIM BOX
# ============================================================

MIN_DENSITY = 0.8

candidates = traj[traj["density"] >= MIN_DENSITY]

if len(candidates) > 0:
    selected_box_index = int(candidates["coverage"].idxmax())
else:
    selected_box_index = int(traj["density"].idxmax())
    print(
        f"No box reached density >= {MIN_DENSITY}. "
        f"Using highest-density box instead."
    )

print("Selected box index:", selected_box_index)
print(traj.loc[selected_box_index, ["coverage", "density", "mass"]])

box.select(selected_box_index)

In [ ]:
# ============================================================
# INSPECT SELECTED BOX
# ============================================================
box.inspect(style="table")

In [ ]:
# ============================================================
# OPTIONAL PRIM EVENT: WORST 10% ZAF ABATEMENT BURDEN
# ============================================================

outcome = results[
    POLICY_ID,
    :,
    OBJECTIVES.index("zaf_mean_abatement_burden")
]

# Omdat hogere abatement burden slechter is, nemen we de 90th percentile
threshold = np.percentile(outcome, 90)

print(f"ZAF abatement burden 90th percentile: {threshold:.4f}")

y = outcome > threshold

print(f"Number of high-burden scenarios: {y.sum()} out of {len(y)} ({100*y.mean():.1f}%)")

In [ ]:
# ============================================================
# RUN PRIM FOR WORST 10% ZAF BURDEN
# ============================================================

prim_alg = prim.Prim(X_prim, y)

box = prim_alg.find_box()

box.show_tradeoff()
plt.title(f"PRIM trade-off — worst 10% ZAF burden for {POLICY_LABEL}")
plt.show()

In [ ]:
# ============================================================
# RUN PRIM FOR WORST 10% ZAF BURDEN
# ============================================================

prim_alg = prim.Prim(X_prim, y)

box = prim_alg.find_box()

box.show_tradeoff()
plt.title(f"PRIM trade-off — worst 10% ZAF burden for {POLICY_LABEL}")
plt.show()


In [ ]:
traj = box.peeling_trajectory.copy()
display(traj.tail())

MIN_DENSITY = 0.8

candidates = traj[traj["density"] >= MIN_DENSITY]

if len(candidates) > 0:
    selected_box_index = int(candidates["coverage"].idxmax())
else:
    selected_box_index = int(traj["density"].idxmax())

box.select(selected_box_index)

print("Selected box index:", selected_box_index)
print(traj.loc[selected_box_index, ["coverage", "density", "mass"]])

box.inspect(style="table")

In [ ]:
from ema_workbench.analysis import prim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# ============================================================
# PRIM Analysis voor geselecteerde policies (P22, P44, P93)
# ============================================================

# 1. Definieer de policies
selected_policies = [22,93,44]

# Zorg dat de satisficing logica vers in het geheugen zit
idx_temp   = obj_idx["global_temperature_2100"]
idx_abate  = obj_idx["zaf_mean_abatement_burden"]
idx_damage = obj_idx["zaf_mean_damage_fraction"]
idx_output = obj_idx["zaf_mean_net_output_ratio"]

pass_temp   = results[:, :, idx_temp] <= 2.0
pass_abate  = results[:, :, idx_abate] <= 0.05
pass_damage = results[:, :, idx_damage] <= 0.05
pass_output = results[:, :, idx_output] >= 0.95

# True = Satisficing, False = Failure
pass_all = pass_temp & pass_abate & pass_damage & pass_output

# 2. Input space voor PRIM
# Zoals gevraagd, bevat dit de 'climate_ensemble_index' (0 t/m 999)
x_prim = pd.DataFrame({'climate_ensemble_index': np.arange(1000)})

table_data = []

# 3. Loop door de geselecteerde policies en voer PRIM uit
for pid in selected_policies:
    print(f"\n" + "="*40)
    print(f" Analysing Policy P{pid}")
    print("="*40)

    # y voor PRIM: we willen falen voorspellen, dus Inverteren (~ is NOT)
    # y is True als het scenario faalt voor deze policy
    y_failure = ~(pass_all[pid, :])
    failure_rate = y_failure.mean()

    print(f"Failure rate: {failure_rate:.1%} ({y_failure.sum()} out of 1000 scenarios)")

    if failure_rate == 0.0 or failure_rate == 1.0:
        print("PRIM is niet bruikbaar: de policy faalt altijd of faalt nooit.")
        table_data.append({
            'Policy ID': f'P{pid}',
            'Failure Rate': f'{failure_rate:.1%}',
            'PRIM Density': '-',
            'PRIM Coverage': '-',
            'climate_ensemble_index range': '-',
            'Interpretation': 'Constant outcome'
        })
        continue

    # 4. Voer de PRIM box search uit
    alg = prim.Prim(x_prim, y_failure, peel_alpha=0.1)
    box1 = alg.find_box()

    # Haal de beste (laatste) box op
    best_box_idx = box1.peeling_trajectory.index[-1]
    stats = box1.peeling_trajectory.loc[best_box_idx]

    density = stats['density']
    coverage = stats['coverage']

    # --- DE FIX: box_extents is nu box_lims en we gebruiken .iloc ---
    box_limits_df = box1.box_lims[best_box_idx]
    min_idx = box_limits_df['climate_ensemble_index'].iloc[0]
    max_idx = box_limits_df['climate_ensemble_index'].iloc[1]

    print(f"Best Box - Density: {density:.2f}, Coverage: {coverage:.2f}")
    print(f"Index range: [{min_idx:.0f}, {max_idx:.0f}]")

    # Bouw de tabel rij op
    table_data.append({
        'Policy ID': f'P{pid}',
        'Failure Rate': f'{failure_rate:.1%}',
        'PRIM Density': f'{density:.2f}',
        'PRIM Coverage': f'{coverage:.2f}',
        'climate_ensemble_index range': f'[{min_idx:.0f} - {max_idx:.0f}]',
        'Interpretation': 'Concentrated failures' if density > 0.7 else 'Scattered failures'
    })

# 4. Print de uiteindelijke DataFrame voor in LaTeX
df_prim_table = pd.DataFrame(table_data)
print("\n=== Data for LaTeX Table ===")
display(df_prim_table)

In [ ]:
##klaar/?

ECR mapping

In [ ]:
# ============================================================
# ECR EXTRACTION FOR SELECTED RBF POLICIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from justice.model import JUSTICE
from justice.util.enumerations import (
    Economy,
    DamageFunction,
    Abatement,
    WelfareFunction,
)

from JUSTICE_example import JUSTICE_run_policy_index


# ============================================================
# 1. SETTINGS
# ============================================================

# Pas dit pad aan naar jullie daadwerkelijke RBF / reference set csv
RBF_WEIGHTS_PATH = Path(RESULTS_DIR) / "reference_set_prioritarian_100000.csv"
# Jullie geselecteerde policies
SELECTED_POLICIES = [22, 93, 44]

# Als 22, 93, 44 echte rij-indexen zijn in de csv: laat dit None.
# Als je csv een kolom heeft zoals "policy_id", zet dan POLICY_ID_COL = "policy_id"
POLICY_ID_COL = None

REFERENCE_SCENARIO = 2
SOCIAL_WELFARE_FUNCTION = WelfareFunction.PRIORITARIAN

N_INPUTS_RBF = 2
EMISSION_CONTROL_START_YEAR = 2025
MAX_ANNUAL_GROWTH_RATE = 0.04
MIN_EMISSION_CONTROL_RATE = 0.01

MAX_TEMPERATURE = 16.0
MIN_TEMPERATURE = 0.0
MAX_DIFFERENCE = 2.0
MIN_DIFFERENCE = 0.0


# ============================================================
# 2. HELPER: MAP POLICY ID TO CSV ROW
# ============================================================

def resolve_policy_row(policy_id, rbf_df, policy_id_col=None):
    """
    Returns the correct row index in the RBF csv.

    Use policy_id_col if your csv has an explicit policy_id column.
    Otherwise policy_id is treated as the row index.
    """
    if policy_id_col is not None and policy_id_col in rbf_df.columns:
        matches = rbf_df.index[rbf_df[policy_id_col].astype(int) == int(policy_id)].tolist()

        if len(matches) == 0:
            raise ValueError(f"Policy {policy_id} not found in column {policy_id_col}")

        return matches[0]

    # Otherwise assume policy_id is the row number
    if int(policy_id) < 0 or int(policy_id) >= len(rbf_df):
        raise ValueError(
            f"Policy {policy_id} is outside the csv row range 0-{len(rbf_df)-1}. "
            "If this is a policy_id column, set POLICY_ID_COL."
        )

    return int(policy_id)


# ============================================================
# 3. RUN SELECTED POLICIES AND EXTRACT ECR
# ============================================================

rbf_df = pd.read_csv(RBF_WEIGHTS_PATH)

policy_runs = {}

for policy_id in SELECTED_POLICIES:
    print(f"Running policy {policy_id}...")

    # Important because JUSTICE caches the model instance.
    JUSTICE.hard_reset()

    model = JUSTICE(
        scenario=REFERENCE_SCENARIO,
        economy_type=Economy.NEOCLASSICAL,
        damage_function_type=DamageFunction.KALKUHL,
        abatement_type=Abatement.ENERDATA,
        social_welfare_function=SOCIAL_WELFARE_FUNCTION,
    )

    time_horizon = model.time_horizon
    data_loader = model.data_loader

    emission_start_ts = time_horizon.year_to_timestep(
        EMISSION_CONTROL_START_YEAR,
        timestep=1,
    )

    policy_row = resolve_policy_row(
        policy_id=policy_id,
        rbf_df=rbf_df,
        policy_id_col=POLICY_ID_COL,
    )

    datasets = JUSTICE_run_policy_index(
        model=model,
        path_to_rbf_weights=str(RBF_WEIGHTS_PATH),
        rbf_policy_index=policy_row,
        time_horizon=time_horizon,
        data_loader=data_loader,
        n_inputs_rbf=N_INPUTS_RBF,
        max_annual_growth_rate=MAX_ANNUAL_GROWTH_RATE,
        emission_control_start_timestep=emission_start_ts,
        min_emission_control_rate=MIN_EMISSION_CONTROL_RATE,
        allow_emission_fallback=False,
        endogenous_savings_rate=True,
        max_temperature=MAX_TEMPERATURE,
        min_temperature=MIN_TEMPERATURE,
        max_difference=MAX_DIFFERENCE,
        min_difference=MIN_DIFFERENCE,
    )

    policy_runs[policy_id] = {
        "datasets": datasets,
        "model": model,
        "policy_row": policy_row,
    }

print("Done.")

In [ ]:

#EXTRACT SOUTH AFRICA ECR TRAJECTORIES

# Pak model-info uit één run
example_model = next(iter(policy_runs.values()))["model"]
regions = np.array(example_model.data_loader.REGION_LIST)
years = np.array(example_model.time_horizon.model_time_horizon)

# Zoek zaf-index
zaf_idx = np.where(np.char.lower(regions.astype(str)) == "zaf")[0][0]

zaf_ecr_summary = []

for policy_id, run in policy_runs.items():
    ds = run["datasets"]

    # Shape: regions x timesteps x ensembles
    ecr = ds["constrained_emission_control_rate"]

    # South Africa: timesteps x ensembles
    zaf_ecr = ecr[zaf_idx, :, :]

    median = np.nanmedian(zaf_ecr, axis=1)
    p10 = np.nanpercentile(zaf_ecr, 10, axis=1)
    p90 = np.nanpercentile(zaf_ecr, 90, axis=1)

    for year in [2030, 2050, 2100]:
        t = example_model.time_horizon.year_to_timestep(year, timestep=1)

        zaf_ecr_summary.append({
            "policy": policy_id,
            "year": year,
            "zaf_ecr_median": median[t],
            "zaf_ecr_p10": p10[t],
            "zaf_ecr_p90": p90[t],
        })

zaf_ecr_summary = pd.DataFrame(zaf_ecr_summary)
display(zaf_ecr_summary)

In [ ]:
# ============================================================
# 5. PLOT SOUTH AFRICA ECR TRAJECTORIES
# ============================================================

plt.figure(figsize=(9, 5))

for policy_id, run in policy_runs.items():
    ds = run["datasets"]
    ecr = ds["constrained_emission_control_rate"]
    zaf_ecr = ecr[zaf_idx, :, :]

    median = np.nanmedian(zaf_ecr, axis=1)
    p10 = np.nanpercentile(zaf_ecr, 10, axis=1)
    p90 = np.nanpercentile(zaf_ecr, 90, axis=1)

    plt.plot(years, median, label=f"Policy {policy_id}")
    plt.fill_between(years, p10, p90, alpha=0.15)

plt.axvline(2025, linestyle="--", linewidth=1, label="ECR constraint start")
plt.xlim(2015, 2100)
plt.ylim(0, 1.05)
plt.xlabel("Year")
plt.ylabel("South Africa emission control rate μ")
plt.title("South Africa emission control rate trajectories for selected RBF policies")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 6. SOUTH AFRICA ABATEMENT BURDEN TRAJECTORIES
# ============================================================

plt.figure(figsize=(9, 5))

for policy_id, run in policy_runs.items():
    ds = run["datasets"]

    abatement = ds["abatement_cost"]
    gross_output = ds["gross_economic_output"]

    burden = np.divide(
        abatement,
        gross_output,
        out=np.full_like(abatement, np.nan),
        where=gross_output != 0,
    )

    zaf_burden = burden[zaf_idx, :, :]

    median = np.nanmedian(zaf_burden, axis=1)
    p10 = np.nanpercentile(zaf_burden, 10, axis=1)
    p90 = np.nanpercentile(zaf_burden, 90, axis=1)

    plt.plot(years, median, label=f"Policy {policy_id}")
    plt.fill_between(years, p10, p90, alpha=0.15)

plt.xlim(2015, 2100)
plt.xlabel("Year")
plt.ylabel("South Africa abatement burden\nabatement cost / gross economic output")
plt.title("South Africa abatement burden for selected RBF policies")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7. REGIONAL ECR AND ABATEMENT BURDEN IN 2100
# ============================================================

target_year = 2100
t_target = example_model.time_horizon.year_to_timestep(target_year, timestep=1)

regional_records = []

for policy_id, run in policy_runs.items():
    ds = run["datasets"]

    ecr = ds["constrained_emission_control_rate"]

    abatement = ds["abatement_cost"]
    gross_output = ds["gross_economic_output"]

    burden = np.divide(
        abatement,
        gross_output,
        out=np.full_like(abatement, np.nan),
        where=gross_output != 0,
    )

    for r_idx, region in enumerate(regions):
        regional_records.append({
            "policy": policy_id,
            "region": str(region),
            "year": target_year,
            "ecr_median": np.nanmedian(ecr[r_idx, t_target, :]),
            "abatement_burden_median": np.nanmedian(burden[r_idx, t_target, :]),
        })

regional_df = pd.DataFrame(regional_records)

display(
    regional_df
    .sort_values(["policy", "abatement_burden_median"], ascending=[True, False])
    .head(30)
)

In [ ]:
# ============================================================
# THREE WORLD MAPS IN ONE FIGURE
# For selected policies: 22, 93, 44
# Works AFTER policy_runs has already been created
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

SELECTED_POLICIES = [22, 93, 44]
MAP_YEAR = 2100

# Choose:
# "abatement_burden"  -> abatement_cost / gross_economic_output
# "ecr"               -> constrained emission control rate
MAP_VALUE = "abatement_burden"

# Optional: save figure
SAVE_FIG = True
FIG_NAME = f"three_world_maps_{MAP_VALUE}_{MAP_YEAR}.png"

# ------------------------------------------------------------
# 2. BASIC CHECKS
# ------------------------------------------------------------

assert "policy_runs" in globals(), (
    "policy_runs not found. Run the ECR extraction cell first."
)

for pid in SELECTED_POLICIES:
    assert pid in policy_runs, f"Policy {pid} not found in policy_runs"

# ------------------------------------------------------------
# 3. GET MODEL INFO
# ------------------------------------------------------------

example_model = next(iter(policy_runs.values()))["model"]
regions = np.array(example_model.data_loader.REGION_LIST).astype(str)
years = np.array(example_model.time_horizon.model_time_horizon)

# Get timestep for selected year
t_map = example_model.time_horizon.year_to_timestep(MAP_YEAR, timestep=1)

# ------------------------------------------------------------
# 4. HELPER FUNCTION TO EXTRACT REGIONAL METRIC
# ------------------------------------------------------------

def get_regional_metric_for_policy(policy_id, year=2100, value="abatement_burden"):
    """
    Returns dataframe with one regional value per JUSTICE region for one policy.

    Output columns:
        region
        iso_a3
        value
        policy
    """
    run = policy_runs[policy_id]
    ds = run["datasets"]

    t = example_model.time_horizon.year_to_timestep(year, timestep=1)

    if value == "abatement_burden":
        abatement = ds["abatement_cost"]              # shape: region x time x ensemble
        gross_output = ds["gross_economic_output"]    # shape: region x time x ensemble

        metric = np.divide(
            abatement,
            gross_output,
            out=np.full_like(abatement, np.nan, dtype=float),
            where=gross_output != 0,
        )

    elif value == "ecr":
        metric = ds["constrained_emission_control_rate"]

    else:
        raise ValueError("value must be 'abatement_burden' or 'ecr'")

    # Median over ensembles for the selected year
    regional_values = np.nanmedian(metric[:, t, :], axis=1)

    df = pd.DataFrame({
        "region": regions,
        "iso_a3": [r.upper() for r in regions],
        "value": regional_values,
        "policy": policy_id,
    })

    return df

# ------------------------------------------------------------
# 5. BUILD DATAFRAMES FOR ALL POLICIES
# ------------------------------------------------------------

map_dfs = {}
all_values = []

for pid in SELECTED_POLICIES:
    df_map = get_regional_metric_for_policy(
        policy_id=pid,
        year=MAP_YEAR,
        value=MAP_VALUE,
    )
    map_dfs[pid] = df_map
    all_values.extend(df_map["value"].replace([np.inf, -np.inf], np.nan).dropna().tolist())

all_values = np.array(all_values, dtype=float)

# Shared color scale across all 3 maps
vmin = np.nanmin(all_values)
vmax = np.nanmax(all_values)

print(f"Shared color scale: vmin={vmin:.6f}, vmax={vmax:.6f}")

# ------------------------------------------------------------
# 6. LOAD WORLD GEOMETRY
# Fixed version for GeoPandas >= 1.0
# ------------------------------------------------------------

import geopandas as gpd

# Natural Earth country polygons, same dataset idea as old naturalearth_lowres
WORLD_URL = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"

world_raw = gpd.read_file(WORLD_URL)

print("Available columns in world map:")
print(world_raw.columns.tolist())

# Standardize column names
# Natural Earth usually has NAME and ISO_A3
world = world_raw.copy()

if "ISO_A3" in world.columns:
    iso_col = "ISO_A3"
elif "ADM0_A3" in world.columns:
    iso_col = "ADM0_A3"
elif "iso_a3" in world.columns:
    iso_col = "iso_a3"
else:
    raise ValueError("Could not find ISO-3 country code column in world map data.")

if "NAME" in world.columns:
    name_col = "NAME"
elif "ADMIN" in world.columns:
    name_col = "ADMIN"
elif "name" in world.columns:
    name_col = "name"
else:
    raise ValueError("Could not find country name column in world map data.")

world = world[[name_col, iso_col, "geometry"]].copy()
world = world.rename(columns={name_col: "name", iso_col: "iso_a3"})

world["iso_a3"] = world["iso_a3"].astype(str).str.upper()

# Remove invalid Natural Earth placeholders if present
world = world[world["iso_a3"] != "-99"].copy()

print("World map loaded successfully.")
print(world.head())
# ------------------------------------------------------------
# 7. CHECK WHICH REGION CODES MATCH THE WORLD MAP
# ------------------------------------------------------------

world_codes = set(world["iso_a3"].astype(str).str.upper())
justice_codes = set([r.upper() for r in regions])

matched_codes = sorted(justice_codes.intersection(world_codes))
unmatched_codes = sorted(justice_codes.difference(world_codes))

print(f"Matched region codes: {len(matched_codes)}")
print(f"Unmatched region codes: {len(unmatched_codes)}")
if len(unmatched_codes) > 0:
    print("Unmatched codes:")
    print(unmatched_codes)
# ------------------------------------------------------------
# 8. PLOT THREE MAPS IN ONE FIGURE
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
axes = axes.flatten()

cmap = "OrRd"
norm = Normalize(vmin=vmin, vmax=vmax)

if MAP_VALUE == "abatement_burden":
    figure_title = f"Regional abatement burden in {MAP_YEAR}"
    colorbar_label = "Abatement burden\n(abatement cost / gross economic output)"
elif MAP_VALUE == "ecr":
    figure_title = f"Regional emission control rate in {MAP_YEAR}"
    colorbar_label = "Emission control rate"
else:
    figure_title = f"Regional metric in {MAP_YEAR}"
    colorbar_label = "Value"

for ax, pid in zip(axes, SELECTED_POLICIES):
    df_map = map_dfs[pid]

    # Merge model outputs onto world geometries
    plot_gdf = world.merge(df_map, on="iso_a3", how="left")

    # Plot the map
    plot_gdf.plot(
        column="value",
        cmap=cmap,
        linewidth=0.4,
        edgecolor="black",
        ax=ax,
        missing_kwds={
            "color": "lightgrey",
            "edgecolor": "white",
            "label": "No data"
        },
        vmin=vmin,
        vmax=vmax,
    )

    # Highlight South Africa if present
    zaf = plot_gdf[plot_gdf["iso_a3"] == "ZAF"]
    if not zaf.empty:
        zaf.boundary.plot(ax=ax, linewidth=1.8, edgecolor="blue")

    ax.set_title(f"Policy {pid}", fontsize=12)
    ax.axis("off")

# Shared colorbar
sm = ScalarMappable(norm=norm, cmap=cmap)
sm._A = []
cbar = fig.colorbar(sm, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label(colorbar_label)

fig.suptitle(figure_title, fontsize=16, y=0.98)
plt.tight_layout()

# Save if wanted
if SAVE_FIG:
    plt.savefig(FIG_NAME, dpi=300, bbox_inches="tight")
    print(f"Figure saved as: {FIG_NAME}")

plt.show()

# ------------------------------------------------------------
# 9. OPTIONAL: COMBINED TABLE USED FOR THE MAPS
# ------------------------------------------------------------

combined_map_df = pd.concat([map_dfs[pid] for pid in SELECTED_POLICIES], ignore_index=True)
display(combined_map_df.head(20))

In [ ]:
# ============================================================
# THREE WORLD MAPS IN ONE FIGURE USING RICE50 REGION MAPPING
# For selected policies: 22, 93, 44
# Works AFTER policy_runs has already been created
# ============================================================

import os
import json
import pathlib
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd


# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

SELECTED_POLICIES = [22, 93, 44]
MAP_YEAR = 2100

# Choose:
# "abatement_burden" = abatement_cost / gross_economic_output
# "ecr"              = constrained emission control rate
MAP_VALUE = "abatement_burden"

SAVE_FIG = True

plot_path = os.path.join(
    PLOTS_DIR,
    f"rice50_world_map_3_policies_{MAP_VALUE}_{MAP_YEAR}.png"
)


# ------------------------------------------------------------
# 2. BASIC CHECKS
# ------------------------------------------------------------

assert "policy_runs" in globals(), (
    "policy_runs not found. Run the ECR extraction cell first."
)

for pid in SELECTED_POLICIES:
    assert pid in policy_runs, f"Policy {pid} not found in policy_runs"


# ------------------------------------------------------------
# 3. GET MODEL INFO
# ------------------------------------------------------------

example_model = next(iter(policy_runs.values()))["model"]

REGION_LIST = np.array(example_model.data_loader.REGION_LIST).astype(str)
REGION_LIST = np.array([r.lower() for r in REGION_LIST])

N_REGIONS = len(REGION_LIST)

_time_horizon = example_model.time_horizon
t_map = _time_horizon.year_to_timestep(MAP_YEAR, timestep=1)

print(f"Map snapshot year: {MAP_YEAR}")
print(f"Timestep: {t_map}")
print(f"Number of JUSTICE regions: {N_REGIONS}")
print("First regions:", REGION_LIST[:10])


# ------------------------------------------------------------
# 4. FIND JUSTICE ROOT AND LOAD RICE50 MAPPING
# ------------------------------------------------------------

_rice50_dict_path = os.path.join(
    str(_JUSTICE_ROOT),
    "data",
    "input",
    "rice50_regions_dict.json"
)

assert os.path.exists(_rice50_dict_path), (
    f"Could not find rice50_regions_dict.json at:\n{_rice50_dict_path}\n"
    "If this path is wrong, set _JUSTICE_ROOT to your JUSTICE project folder."
)

with open(_rice50_dict_path, "r") as f:
    _rice50_dict = json.load(f)

# Map country ISO3 code -> RICE50 region
iso_to_rice50 = {
    iso.upper(): region.lower()
    for region, isos in _rice50_dict.items()
    for iso in isos
}

# Some Natural Earth country names need fallback mapping
_name_fallback = {
    "France":     "fra",
    "Norway":     "nor",
    "Kosovo":     "oeu",
    "N. Cyprus":  "tur",
    "Somaliland": "rsaf",
}


# ------------------------------------------------------------
# 5. EXTRACT REGIONAL SNAPSHOT VALUES FROM POLICY RUNS
# ------------------------------------------------------------

def _snapshot_values(arr, timestep):
    """
    Extract values for one timestep.

    Accepts:
    - 3D array: region x time x ensemble
    - 2D array: region x time
    """
    arr = np.asarray(arr)

    if arr.ndim == 3:
        return np.nanmedian(arr[:, timestep, :], axis=1)

    if arr.ndim == 2:
        return arr[:, timestep]

    raise ValueError(f"Expected 2D or 3D array, got shape {arr.shape}")


def get_region_metric_snapshot(policy_id, timestep, value="abatement_burden"):
    """
    Returns dictionary:
        RICE50 region name -> metric value
    """
    ds = policy_runs[policy_id]["datasets"]

    if value == "abatement_burden":
        abatement = np.asarray(ds["abatement_cost"], dtype=float)
        gross_output = np.asarray(ds["gross_economic_output"], dtype=float)

        metric = np.divide(
            abatement,
            gross_output,
            out=np.full_like(abatement, np.nan, dtype=float),
            where=gross_output != 0,
        )

    elif value == "ecr":
        metric = np.asarray(ds["constrained_emission_control_rate"], dtype=float)

    else:
        raise ValueError("MAP_VALUE must be 'abatement_burden' or 'ecr'.")

    values = _snapshot_values(metric, timestep)

    if len(values) != N_REGIONS:
        raise ValueError(
            f"Metric has {len(values)} regions, but REGION_LIST has {N_REGIONS}."
        )

    return {
        REGION_LIST[i]: values[i]
        for i in range(N_REGIONS)
    }


region_metric_by_policy = {}

for pid in SELECTED_POLICIES:
    region_metric_by_policy[pid] = get_region_metric_snapshot(
        policy_id=pid,
        timestep=t_map,
        value=MAP_VALUE,
    )


# ------------------------------------------------------------
# 6. LOAD WORLD SHAPEFILE
# ------------------------------------------------------------

# First try the local Natural Earth file shipped with pyogrio.
# This avoids the GeoPandas 1.0 removed-dataset issue.
world = None

try:
    pyogrio_spec = importlib.util.find_spec("pyogrio")
    if pyogrio_spec is not None:
        _pyogrio_path = pathlib.Path(pyogrio_spec.origin).parent
        _ne_shp = (
            _pyogrio_path
            / "tests"
            / "fixtures"
            / "naturalearth_lowres"
            / "naturalearth_lowres.shp"
        )

        if _ne_shp.exists():
            world = gpd.read_file(str(_ne_shp))
            print("Loaded Natural Earth map from pyogrio fixture.")

except Exception as e:
    print("Could not load pyogrio fixture:", e)

# Fallback: read Natural Earth directly from URL
if world is None:
    WORLD_URL = (
        "https://naturalearth.s3.amazonaws.com/"
        "110m_cultural/ne_110m_admin_0_countries.zip"
    )
    world = gpd.read_file(WORLD_URL)
    print("Loaded Natural Earth map from online URL.")


# Standardize column names
world = world.copy()

if "iso_a3" not in world.columns:
    if "ISO_A3" in world.columns:
        world = world.rename(columns={"ISO_A3": "iso_a3"})
    elif "ADM0_A3" in world.columns:
        world = world.rename(columns={"ADM0_A3": "iso_a3"})
    else:
        raise ValueError("Could not find ISO3 column in world shapefile.")

if "name" not in world.columns:
    if "NAME" in world.columns:
        world = world.rename(columns={"NAME": "name"})
    elif "ADMIN" in world.columns:
        world = world.rename(columns={"ADMIN": "name"})
    else:
        raise ValueError("Could not find country name column in world shapefile.")

world["iso_a3"] = world["iso_a3"].astype(str).str.upper()


# ------------------------------------------------------------
# 7. MAP WORLD COUNTRIES TO RICE50 REGIONS
# ------------------------------------------------------------

world["rice50"] = world["iso_a3"].map(iso_to_rice50)

mask_missing = world["rice50"].isna()
world.loc[mask_missing, "rice50"] = world.loc[mask_missing, "name"].map(_name_fallback)

n_mapped = world["rice50"].notna().sum()
print(f"Countries mapped to RICE50 regions: {n_mapped}/{len(world)}")

unmapped = world.loc[world["rice50"].isna(), ["name", "iso_a3"]]
if len(unmapped) > 0:
    print("Unmapped countries:")
    display(unmapped.head(30))


# ------------------------------------------------------------
# 8. ADD POLICY VALUES TO COUNTRY MAP
# ------------------------------------------------------------

metric_cols = []

for pid in SELECTED_POLICIES:
    col = f"{MAP_VALUE}_policy_{pid}"
    metric_cols.append(col)

    world[col] = world["rice50"].map(region_metric_by_policy[pid])


# ------------------------------------------------------------
# 9. DISSOLVE COUNTRIES INTO RICE50 REGIONS
# ------------------------------------------------------------

regions_gdf = (
    world[world["rice50"].notna()]
    .dissolve(by="rice50", aggfunc="first")
    .reset_index()
)

regions_gdf = regions_gdf[["rice50", "geometry"] + metric_cols].copy()
regions_gdf = gpd.GeoDataFrame(regions_gdf, geometry="geometry", crs=world.crs)

print(f"Dissolved to {len(regions_gdf)} RICE50 regions")
display(regions_gdf.head())


# ------------------------------------------------------------
# 10. SHARED COLOR SCALE
# ------------------------------------------------------------

all_vals = pd.concat(
    [regions_gdf[col].replace([np.inf, -np.inf], np.nan).dropna()
     for col in metric_cols],
    ignore_index=True,
)

vmin_all = all_vals.min()
vmax_all = all_vals.max()

print(f"Shared color scale: {vmin_all:.6f} to {vmax_all:.6f}")

_cmap = plt.cm.YlOrRd
_norm = mcolors.Normalize(vmin=vmin_all, vmax=vmax_all)


# ------------------------------------------------------------
# 11. PLOT THREE MAPS IN ONE FIGURE
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
axes = axes.flatten()

if MAP_VALUE == "abatement_burden":
    main_title = f"Regional abatement burden in {MAP_YEAR}"
    cbar_label = "Abatement burden\nabatement cost / gross economic output"
elif MAP_VALUE == "ecr":
    main_title = f"Regional emission control rate in {MAP_YEAR}"
    cbar_label = "Emission control rate (0–1)"
else:
    main_title = f"Regional values in {MAP_YEAR}"
    cbar_label = "Value"


for ax, pid, col in zip(axes, SELECTED_POLICIES, metric_cols):
    # Grey country background
    world.plot(ax=ax, color="0.88", edgecolor="white", linewidth=0.2)

    # RICE50 regional values
    regions_gdf.plot(
        ax=ax,
        column=col,
        cmap=_cmap,
        norm=_norm,
        edgecolor="0.35",
        linewidth=0.45,
        missing_kwds={
            "color": "0.88",
            "edgecolor": "white",
            "label": "No data",
        },
    )

    # Highlight South Africa
    zaf = regions_gdf[regions_gdf["rice50"].str.lower() == "zaf"]
    if not zaf.empty:
        zaf.boundary.plot(ax=ax, color="blue", linewidth=2.0)

    ax.set_title(f"Policy {pid}", fontsize=13, fontweight="bold")
    ax.set_axis_off()


# Make space for one shared colorbar on the right
fig.subplots_adjust(right=0.90, wspace=0.05)

cbar_ax = fig.add_axes([0.92, 0.22, 0.015, 0.56])
sm = plt.cm.ScalarMappable(cmap=_cmap, norm=_norm)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label(cbar_label, fontsize=10)

fig.suptitle(
    f"{main_title} — JUSTICE RICE50 regions\n"
    f"Selected policies {SELECTED_POLICIES}",
    fontsize=15,
    y=0.98,
)

if SAVE_FIG:
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    print("Figure saved:", plot_path)

plt.show()

In [ ]:
# ============================================================
# THREE WORLD MAPS IN ONE FIGURE USING RICE50 REGION MAPPING
# For selected policies: 22, 93, 44
# ECR
# ============================================================

import os
import json
import pathlib
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd


# ------------------------------------------------------------
# 1. SETTINGS
# ------------------------------------------------------------

SELECTED_POLICIES = [22, 93, 44]
MAP_YEAR = 2100

# Choose:
# "abatement_burden" = abatement_cost / gross_economic_output
# "ecr"              = constrained emission control rate
MAP_VALUE = "ECR"

SAVE_FIG = True


plot_path = os.path.join(
    PLOTS_DIR,
    f"rice50_world_map_3_policies_{MAP_VALUE}_{MAP_YEAR}.png"
)


# ------------------------------------------------------------
# 2. BASIC CHECKS
# ------------------------------------------------------------

assert "policy_runs" in globals(), (
    "policy_runs not found. Run the ECR extraction cell first."
)

for pid in SELECTED_POLICIES:
    assert pid in policy_runs, f"Policy {pid} not found in policy_runs"


# ------------------------------------------------------------
# 3. GET MODEL INFO
# ------------------------------------------------------------

example_model = next(iter(policy_runs.values()))["model"]

REGION_LIST = np.array(example_model.data_loader.REGION_LIST).astype(str)
REGION_LIST = np.array([r.lower() for r in REGION_LIST])

N_REGIONS = len(REGION_LIST)

_time_horizon = example_model.time_horizon
t_map = _time_horizon.year_to_timestep(MAP_YEAR, timestep=1)

print(f"Map snapshot year: {MAP_YEAR}")
print(f"Timestep: {t_map}")
print(f"Number of JUSTICE regions: {N_REGIONS}")
print("First regions:", REGION_LIST[:10])


# ------------------------------------------------------------
# 4. FIND JUSTICE ROOT AND LOAD RICE50 MAPPING
# ------------------------------------------------------------

_rice50_dict_path = os.path.join(
    str(_JUSTICE_ROOT),
    "data",
    "input",
    "rice50_regions_dict.json"
)

assert os.path.exists(_rice50_dict_path), (
    f"Could not find rice50_regions_dict.json at:\n{_rice50_dict_path}\n"
    "If this path is wrong, set _JUSTICE_ROOT to your JUSTICE project folder."
)

with open(_rice50_dict_path, "r") as f:
    _rice50_dict = json.load(f)

# Map country ISO3 code -> RICE50 region
iso_to_rice50 = {
    iso.upper(): region.lower()
    for region, isos in _rice50_dict.items()
    for iso in isos
}

# Some Natural Earth country names need fallback mapping
_name_fallback = {
    "France":     "fra",
    "Norway":     "nor",
    "Kosovo":     "oeu",
    "N. Cyprus":  "tur",
    "Somaliland": "rsaf",
}


# ------------------------------------------------------------
# 5. EXTRACT REGIONAL SNAPSHOT VALUES FROM POLICY RUNS
# ------------------------------------------------------------

def _snapshot_values(arr, timestep):
    """
    Extract values for one timestep.

    Accepts:
    - 3D array: region x time x ensemble
    - 2D array: region x time
    """
    arr = np.asarray(arr)

    if arr.ndim == 3:
        return np.nanmedian(arr[:, timestep, :], axis=1)

    if arr.ndim == 2:
        return arr[:, timestep]

    raise ValueError(f"Expected 2D or 3D array, got shape {arr.shape}")


def get_region_metric_snapshot(policy_id, timestep, value="abatement_burden"):
    """
    Returns dictionary:
        RICE50 region name -> metric value
    """
    ds = policy_runs[policy_id]["datasets"]

    if value == "abatement_burden":
        abatement = np.asarray(ds["abatement_cost"], dtype=float)
        gross_output = np.asarray(ds["gross_economic_output"], dtype=float)

        metric = np.divide(
            abatement,
            gross_output,
            out=np.full_like(abatement, np.nan, dtype=float),
            where=gross_output != 0,
        )

    elif value == "ecr":
        metric = np.asarray(ds["constrained_emission_control_rate"], dtype=float)

    else:
        raise ValueError("MAP_VALUE must be 'abatement_burden' or 'ecr'.")

    values = _snapshot_values(metric, timestep)

    if len(values) != N_REGIONS:
        raise ValueError(
            f"Metric has {len(values)} regions, but REGION_LIST has {N_REGIONS}."
        )

    return {
        REGION_LIST[i]: values[i]
        for i in range(N_REGIONS)
    }


region_metric_by_policy = {}

for pid in SELECTED_POLICIES:
    region_metric_by_policy[pid] = get_region_metric_snapshot(
        policy_id=pid,
        timestep=t_map,
        value=MAP_VALUE,
    )


# ------------------------------------------------------------
# 6. LOAD WORLD SHAPEFILE
# ------------------------------------------------------------

# First try the local Natural Earth file shipped with pyogrio.
# This avoids the GeoPandas 1.0 removed-dataset issue.
world = None

try:
    pyogrio_spec = importlib.util.find_spec("pyogrio")
    if pyogrio_spec is not None:
        _pyogrio_path = pathlib.Path(pyogrio_spec.origin).parent
        _ne_shp = (
            _pyogrio_path
            / "tests"
            / "fixtures"
            / "naturalearth_lowres"
            / "naturalearth_lowres.shp"
        )

        if _ne_shp.exists():
            world = gpd.read_file(str(_ne_shp))
            print("Loaded Natural Earth map from pyogrio fixture.")

except Exception as e:
    print("Could not load pyogrio fixture:", e)

# Fallback: read Natural Earth directly from URL
if world is None:
    WORLD_URL = (
        "https://naturalearth.s3.amazonaws.com/"
        "110m_cultural/ne_110m_admin_0_countries.zip"
    )
    world = gpd.read_file(WORLD_URL)
    print("Loaded Natural Earth map from online URL.")


# Standardize column names
world = world.copy()

if "iso_a3" not in world.columns:
    if "ISO_A3" in world.columns:
        world = world.rename(columns={"ISO_A3": "iso_a3"})
    elif "ADM0_A3" in world.columns:
        world = world.rename(columns={"ADM0_A3": "iso_a3"})
    else:
        raise ValueError("Could not find ISO3 column in world shapefile.")

if "name" not in world.columns:
    if "NAME" in world.columns:
        world = world.rename(columns={"NAME": "name"})
    elif "ADMIN" in world.columns:
        world = world.rename(columns={"ADMIN": "name"})
    else:
        raise ValueError("Could not find country name column in world shapefile.")

world["iso_a3"] = world["iso_a3"].astype(str).str.upper()


# ------------------------------------------------------------
# 7. MAP WORLD COUNTRIES TO RICE50 REGIONS
# ------------------------------------------------------------

world["rice50"] = world["iso_a3"].map(iso_to_rice50)

mask_missing = world["rice50"].isna()
world.loc[mask_missing, "rice50"] = world.loc[mask_missing, "name"].map(_name_fallback)

n_mapped = world["rice50"].notna().sum()
print(f"Countries mapped to RICE50 regions: {n_mapped}/{len(world)}")

unmapped = world.loc[world["rice50"].isna(), ["name", "iso_a3"]]
if len(unmapped) > 0:
    print("Unmapped countries:")
    display(unmapped.head(30))


# ------------------------------------------------------------
# 8. ADD POLICY VALUES TO COUNTRY MAP
# ------------------------------------------------------------

metric_cols = []

for pid in SELECTED_POLICIES:
    col = f"{MAP_VALUE}_policy_{pid}"
    metric_cols.append(col)

    world[col] = world["rice50"].map(region_metric_by_policy[pid])


# ------------------------------------------------------------
# 9. DISSOLVE COUNTRIES INTO RICE50 REGIONS
# ------------------------------------------------------------

regions_gdf = (
    world[world["rice50"].notna()]
    .dissolve(by="rice50", aggfunc="first")
    .reset_index()
)

regions_gdf = regions_gdf[["rice50", "geometry"] + metric_cols].copy()
regions_gdf = gpd.GeoDataFrame(regions_gdf, geometry="geometry", crs=world.crs)

print(f"Dissolved to {len(regions_gdf)} RICE50 regions")
display(regions_gdf.head())


# ------------------------------------------------------------
# 10. SHARED COLOR SCALE
# ------------------------------------------------------------

all_vals = pd.concat(
    [regions_gdf[col].replace([np.inf, -np.inf], np.nan).dropna()
     for col in metric_cols],
    ignore_index=True,
)

vmin_all = all_vals.min()
vmax_all = all_vals.max()

print(f"Shared color scale: {vmin_all:.6f} to {vmax_all:.6f}")

_cmap = plt.cm.YlOrRd
_norm = mcolors.Normalize(vmin=vmin_all, vmax=vmax_all)


# ------------------------------------------------------------
# 11. PLOT THREE MAPS IN ONE FIGURE
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(22, 6))
axes = axes.flatten()

if MAP_VALUE == "abatement_burden":
    main_title = f"Regional abatement burden in {MAP_YEAR}"
    cbar_label = "Abatement burden\nabatement cost / gross economic output"
elif MAP_VALUE == "ecr":
    main_title = f"Regional emission control rate in {MAP_YEAR}"
    cbar_label = "Emission control rate (0–1)"
else:
    main_title = f"Regional values in {MAP_YEAR}"
    cbar_label = "Value"


for ax, pid, col in zip(axes, SELECTED_POLICIES, metric_cols):
    # Grey country background
    world.plot(ax=ax, color="0.88", edgecolor="white", linewidth=0.2)

    # RICE50 regional values
    regions_gdf.plot(
        ax=ax,
        column=col,
        cmap=_cmap,
        norm=_norm,
        edgecolor="0.35",
        linewidth=0.45,
        missing_kwds={
            "color": "0.88",
            "edgecolor": "white",
            "label": "No data",
        },
    )

    # Highlight South Africa
    zaf = regions_gdf[regions_gdf["rice50"].str.lower() == "zaf"]
    if not zaf.empty:
        zaf.boundary.plot(ax=ax, color="blue", linewidth=2.0)

    ax.set_title(f"Policy {pid}", fontsize=13, fontweight="bold")
    ax.set_axis_off()


# Make space for one shared colorbar on the right
fig.subplots_adjust(right=0.90, wspace=0.05)

cbar_ax = fig.add_axes([0.92, 0.22, 0.015, 0.56])
sm = plt.cm.ScalarMappable(cmap=_cmap, norm=_norm)
sm.set_array([])

cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label(cbar_label, fontsize=10)

fig.suptitle(
    f"{main_title} — JUSTICE RICE50 regions\n"
    f"Selected policies {SELECTED_POLICIES}",
    fontsize=15,
    y=0.98,
)

if SAVE_FIG:
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    print("Figure saved:", plot_path)

plt.show()